# NashGAT-MARL: dynamic graph attention and exploitability-aware MARL for traffic-signal control

**Badr El Khalyly**∗ and **Zouhair Ibn Batouta**†<br>
∗ LTIM – Laboratory of Techniques of Information and Modeling, FSBM, Hassan II University, Casablanca, Morocco<br>
† RITM – ESTC/CED, ENSEM, Hassan II University, Casablanca, Morocco<br>
badrelkhalylyphd@gmail.com · Zouhair.ibnbatouta@gmail.com

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/badrelkhalylyphd-boop/NashGAT_MARL_Research/blob/main/notebooks/NashGAT_MARL_Research.ipynb)
Code and released results: https://github.com/badrelkhalylyphd-boop/NashGAT_MARL_Research

**Research notebook.** It is reproducible and self-contained. GPU is optional: the networks are small, so CPU is often as fast.

| Mode | Purpose | Budget |
|---|---|---|
| `REPRODUCE_PAPER` (default) | **reviewers: reproduces the paper's tables and figures.** Downloads the 52 released trained models and logs from GitHub, regenerates every table and figure, and independently re-runs the models and reference controllers to check them | ≈ 15–25 min on Colab CPU |
| `FAST_DEMO` | functional check of the full training pipeline | ≈ 15–30 min; 1 seed, short episodes. **Numbers are NOT research results.** |
| `FULL_EXPERIMENT` | retrains everything from scratch with the paper's protocol | many hours; 5 seeds × 250 episodes × 3,600 s. Retrained numbers differ slightly from the paper (RL training is not bit-reproducible across hardware); conclusions should hold within the confidence intervals |

**What `REPRODUCE_PAPER` guarantees.** The paper's numbers come from the released run files and are regenerated here. Three checks are recomputed in this session. Greedy evaluations of the released models and the FixedTime/MaxPressure references are deterministic, so they must match the paper **exactly**. Stochastic evaluations use a new random stream, so they must agree within sampling noise. The paper's results use the **QueueSim** simulator, so this mode forces QueueSim.

**Integrity rules followed here.** Every number printed is computed in this run; nothing is typed in by hand. Literature metadata was verified against Crossref, OpenAlex, DataCite, arXiv or proceedings pages on 2026-10-19. Scopus could not be accessed, so indexing is marked *requires verification*. Results from **CityFlow** and from the fallback **QueueSim** simulator are separate experimental environments and must never be pooled.

In [ ]:
# ====== experiment mode ======
MODE = "REPRODUCE_PAPER"    # "REPRODUCE_PAPER", "FAST_DEMO" or "FULL_EXPERIMENT"
USE_DRIVE = False           # True: mount Google Drive and copy checkpoints/results there
PREFER_CITYFLOW = True      # try to build CityFlow; fall back to QueueSim if that fails
REPO_URL = "https://github.com/badrelkhalylyphd-boop/NashGAT_MARL_Research.git"

if MODE == "REPRODUCE_PAPER":
    # the paper's protocol; nothing is retrained, the released runs are downloaded and re-checked
    CFG = dict(seeds=[0, 1, 2, 3, 4], episodes=250, horizon=3600, eval_episodes=3, robustness=True,
               br_agents=(0, 1, 5), br_episodes=20, br_eval_episodes=5, gg_steps=1500, gg_seeds=5,
               ablations=["nashgat_no_gnn", "nashgat_no_attention", "nashgat_no_nash",
                          "nashgat_no_comm_penalty", "nashgat_static_graph"])
    PREFER_CITYFLOW = False     # the paper's results were produced with QueueSim
elif MODE == "FAST_DEMO":
    CFG = dict(seeds=[0], episodes=4, horizon=900, eval_episodes=1, robustness=True,
               br_agents=(0, 5), br_episodes=2, br_eval_episodes=2, gg_steps=300, gg_seeds=2,
               ablations=["nashgat_no_nash", "nashgat_static_graph"])
elif MODE == "FULL_EXPERIMENT":
    CFG = dict(seeds=[0, 1, 2, 3, 4], episodes=250, horizon=3600, eval_episodes=3, robustness=True,
               br_agents=(0, 1, 5), br_episodes=20, br_eval_episodes=5, gg_steps=1500, gg_seeds=5,
               ablations=["nashgat_no_gnn", "nashgat_no_attention", "nashgat_no_nash",
                          "nashgat_no_comm_penalty", "nashgat_static_graph"])
else:
    raise ValueError(MODE)
METHODS = ["ippo", "mappo", "gnn_mappo", "nash_mappo", "nashgat"]
print(MODE, CFG)

## Part 1 — Scientific introduction

### 1.1 Problem and motivation
Adaptive signal control is a multi-agent problem. Each intersection controller acts on local sensor data, but its decisions change the queues of its neighbours through finite road storage and spill-back. Graph-attention MARL for traffic signal control (TSC) is mature: CoLight (2019) and many 2025–2026 variants. Almost all of it optimises a **shared** reward. If controllers optimise **their own** queues, which happens with different operators or local retraining, the system is a *general-sum* Markov game. A jointly trained plan can then be *exploitable*: one controller could improve its own reward by deviating unilaterally. This notebook measures that exploitability next to the usual congestion metrics and communication cost.

### 1.2 Recent peer-reviewed or preprint references (2025–2026), all metadata verified
Scopus record-level indexing and journal quartiles: **requires verification**. Scopus access was not available, so neither is claimed. The full list of 49 verified entries is in `references.bib`, which is written in Part 9.

| key | title | venue | year | DOI / URL |
|---|---|---|---|---|
| `shi2026ddfg` | Dynamic Deep Factor Graph for Multi-Agent Reinforcement Learning | IEEE Transactions on Pattern Analysis and Machine Intelligence | 2026 | [10.1109/TPAMI.2025.3634378](https://doi.org/10.1109/TPAMI.2025.3634378) |
| `peng2026gde` | Graph based multi-agent reinforcement learning with evolutionary population for cooperation | Neural Networks | 2026 | [10.1016/j.neunet.2025.108437](https://doi.org/10.1016/j.neunet.2025.108437) |
| `gou2026gsmarl` | A graph-based safe reinforcement learning method for multi-agent cooperation | Neural Networks | 2026 | [10.1016/j.neunet.2026.108693](https://doi.org/10.1016/j.neunet.2026.108693) |
| `liu2025distributednash` | Distributed Nash equilibrium solution for multi-agent game in adversarial environment: A reinforcement learning method | Automatica | 2025 | [10.1016/j.automatica.2025.112342](https://doi.org/10.1016/j.automatica.2025.112342) |
| `zhang2025graphicalgames` | Reinforcement Learning Solutions to Stochastic Multi-Agent Graphical Games With Multiplicative Noise | IEEE Transactions on Circuits and Systems I: Regular Papers | 2025 | [10.1109/TCSI.2025.3548900](https://doi.org/10.1109/TCSI.2025.3548900) |
| `kuang2026llmgncf` | LLM-guided graph neural coordination framework for cooperative multi-agent reinforcement learning | Complex \& Intelligent Systems | 2026 | [10.1007/s40747-026-02356-7](https://doi.org/10.1007/s40747-026-02356-7) |
| `moghaddam2025nashseeking` | Multiagent Reinforcement Learning for Nash Equilibrium Seeking in General-Sum Markov Games | IEEE Transactions on Systems, Man, and Cybernetics: Systems | 2025 | [10.1109/TSMC.2024.3462762](https://doi.org/10.1109/TSMC.2024.3462762) |
| `shang2025magsac` | Graph-Based Cooperation Multiagent Reinforcement Learning for Intelligent Traffic Signal Control | IEEE Internet of Things Journal | 2025 | [10.1109/JIOT.2025.3525640](https://doi.org/10.1109/JIOT.2025.3525640) |
| `zhai2025hgatmarl` | HGAT and Multi-Agent RL-Based Method for Multi-Intersection Traffic Signal Control | IEEE Transactions on Intelligent Transportation Systems | 2025 | [10.1109/TITS.2025.3526223](https://doi.org/10.1109/TITS.2025.3526223) |
| `wang2025hypergraphtsc` | Toward Multi-Agent Reinforcement Learning Based Traffic Signal Control Through Spatio-Temporal Hypergraphs | IEEE Transactions on Mobile Computing | 2025 | [10.1109/TMC.2025.3556243](https://doi.org/10.1109/TMC.2025.3556243) |
| `yang2025gatddpg` | Multi-Agent Deep Reinforcement Learning with Graph Attention Network for Traffic Signal Control in Multiple-Intersection Urban Areas | Transportation Research Record: Journal of the Transportation Research Board | 2025 | [10.1177/03611981241297979](https://doi.org/10.1177/03611981241297979) |
| `yang2025storeforward` | Store-and-forward with graph attention: Enhanced multi-agent reinforcement learning for emergency-responsive traffic signal control | Engineering Applications of Artificial Intelligence | 2025 | [10.1016/j.engappai.2025.111602](https://doi.org/10.1016/j.engappai.2025.111602) |
| `yang2026heterodist` | Heterogeneous graph distributional reinforcement learning for out-of-distribution traffic signal control | Expert Systems with Applications | 2026 | [10.1016/j.eswa.2026.132707](https://doi.org/10.1016/j.eswa.2026.132707) |
| `zhang2026sanis` | Distributed multi-agent reinforcement learning with neighborhood-aware spatio-temporal coordination for traffic network signal control | Scientific Reports | 2026 | [10.1038/s41598-026-65802-z](https://doi.org/10.1038/s41598-026-65802-z) |
| `ji2027stackelbergtsc` | Asynchronous multi-agent reinforcement learning for network-wide traffic signal control via Stackelberg game prioritization | Applied Soft Computing | 2027 | [10.1016/j.asoc.2026.116565](https://doi.org/10.1016/j.asoc.2026.116565) |
| `zhang2025tgcnet` | Bridging Training and Execution via Dynamic Directed Graph-Based Communication in Cooperative Multi-Agent Systems | Proceedings of the AAAI Conference on Artificial Intelligence | 2025 | [10.1609/aaai.v39i22.34507](https://doi.org/10.1609/aaai.v39i22.34507) |
| `zhang2026commprotocols` | Learning Efficient Communication Protocols for Multi-Agent Reinforcement Learning | IEEE Transactions on Machine Learning in Communications and Networking | 2026 | [10.1109/TMLCN.2026.3719222](https://doi.org/10.1109/TMLCN.2026.3719222) |
| `alqudah2026nashhgt` | Nash-regularized heterogeneous graph transformer networks for strategic task offloading in 6G edge computing | International Journal of Data and Network Science | 2026 | [10.5267/j.ijdns.2025.9.009](https://doi.org/10.5267/j.ijdns.2025.9.009) |
| `fei2026gatgame` | Graph attention-based distributed game-theoretic framework for elastic collaborative predictive maintenance in communication-constrained production line clusters | Engineering Research Express | 2026 | [10.1088/2631-8695/ae67d6](https://doi.org/10.1088/2631-8695/ae67d6) |
| `li2026evpricing` | Mixed Competitive--Cooperative Pricing for EV Charging Stations: A Multi-Agent Reinforcement Learning Approach With Heterogeneous Hierarchical Attention | IEEE Transactions on Smart Grid | 2026 | [10.1109/TSG.2026.3669058](https://doi.org/10.1109/TSG.2026.3669058) |
| `cheng2026sgasac` | SGA-SAC: Fusing Bayesian Stackelberg games to DRL with graph attention for explainable task offloading in vehicular edge computing | Information Fusion | 2026 | [10.1016/j.inffus.2026.104506](https://doi.org/10.1016/j.inffus.2026.104506) |
| `samanta2026gatmappoeig` | GAT-MAPPO-EIG: A Graph Attention Multi-Agent Reinforcement Learning Framework for Escape Interdiction Games on Dynamic Transportation Networks | Preprint, not peer reviewed | 2026 | [10.21203/rs.3.rs-10945491/v1](https://doi.org/10.21203/rs.3.rs-10945491/v1) |

### 1.3 Research gap and what survives scrutiny
* We found no verified paper that combines dynamic GAT, a MAPPO-style actor-critic, an explicit exploitability regulariser and communication-budgeted execution for TSC. Each pair has precedent, though. The closest works are `fei2026gatgame`, `alqudah2026nashhgt`, the preprint `samanta2026gatmappoeig`, and the Nash-based TSC works `zhang2023nashdqn` and `yan2024dcnpg`. Their full texts must be read before any "first" claim.
* **Key negative result (Part 2, Prop. 4):** the natural differentiable exploitability surrogate with a detached critic has *exactly* the gradient of selfish policy improvement. In that form the "Nash-aware" objective is **reward mixing** between team and individual rewards. Part 4 checks this numerically.

### 1.4 Research questions and hypotheses
* **RQ1.** Does graph attention with traffic-coupling edge features improve congestion metrics over MLP actors under an equal budget?
  *H1:* GNN-MAPPO and NashGAT reach a lower average travel time (ATT) than MAPPO.
* **RQ2.** Does the gap regulariser ($\lambda>0$) reduce measured best-response exploitability, and at what welfare cost?
  *H2:* Nash-MAPPO has a lower sampled NashConv than MAPPO.
* **RQ3.** Can learned gates cut messages while keeping ATT?
  *H3:* NashGAT sends fewer messages than GNN-MAPPO, with no significant ATT loss.
* **RQ4.** Robustness and transfer: noise, link failures, demand shifts, failed controllers, and zero-shot transfer to Hangzhou 4×4.

## Part 2 — Mathematical foundations

### 2.1 Partially observable general-sum Markov game
$\mathcal{G}=\langle\mathcal{N},\mathcal{S},\{\mathcal{O}_i\},\{\mathcal{A}_i\},P,\{r_i\},Z,\rho,\gamma\rangle$. Its components:
* agents $i\in\mathcal{N}$ are the signalised intersections (12 in Jinan 3×4);
* actions $a_i$ are the 8 admissible light phases (a change triggers a 3 s yellow), plus a communication gate $g_i\in\{0,1\}$ for NashGAT;
* the observation $o_i\in\mathbb{R}^{32}$ holds movement queues, approaching counts and the phase one-hot;
* the reward is $r_i=-\frac1{20}\sum_{m\in\mathcal{M}_i}q_m$ (own queues only, hence **general-sum**);
* the discount is $\gamma=0.99$ per 10 s decision.

Returns are $J_i(\pi)=\mathbb{E}_\pi[\sum_t\gamma^tr_i(t)]$, and welfare is $W=\frac1n\sum_iJ_i$.

### 2.2 Graph neural networks and graph attention
Communication graph: $G_t=(V,E_t)$, with $E_t=\{j\!\to\!i\in E_0: g_{j,t-1}=1,\ \text{link up}\}$. One GATv2 round with edge features $x_{ji}$ (road occupancy, queued share) gives
$$e_{ij}=\mathbf a^\top\mathrm{LeakyReLU}(W_sz_j+W_tz_i+W_ex_{ji}),\quad \alpha_{ij}=\mathrm{softmax}_j(e_{ij}),\quad h_i=[z_i\,\|\,z_i+\mathrm{ReLU}(\textstyle\sum_j\alpha_{ij}W_sz_j)].$$

### 2.3 Multi-agent PPO
Shared-parameter actor $\pi_\theta(a_i\mid h_i)$ (decentralised execution) and a centralised critic (training only). The clipped surrogate is
$$L^{\text{PPO}}=-\mathbb{E}\big[\min(\rho_t\hat A_t,\ \mathrm{clip}(\rho_t,1\!\pm\!\epsilon)\hat A_t)\big],\qquad \rho_t=\frac{\pi_\theta(a_t|h_t)}{\pi_{\theta_\text{old}}(a_t|h_t)},$$
with GAE($\lambda$) advantages computed on the team reward (MAPPO family) or the individual reward (IPPO).

### 2.4 Nash equilibrium, best response, exploitability
$\pi^\star$ is a Nash equilibrium in the policy class $\Pi$ if, for all $i$ and all $\pi_i'\in\Pi_i$, $J_i(\pi_i',\pi^\star_{-i})\le J_i(\pi^\star)$. Exploitability (NashConv) is
$$\mathcal{E}(\pi)=\sum_i\big[\max_{\pi_i'\in\Pi_i}J_i(\pi_i',\pi_{-i})-J_i(\pi)\big]\ge0.$$
On the traffic game, $\mathcal{E}$ is estimated with **approximate best responses**: PPO fine-tuning of $\pi_k$ on $r_k$ while the other agents are frozen. This gives a noisy **lower** bound.

### 2.5 Local exploitability gap (critic surrogate)
$$\mathrm{gap}_i(s)=\max_a\hat Q_i(s,a)-\textstyle\sum_a\pi_i(a|h_i)\hat Q_i(s,a)\ \ge0 .$$
* **Prop. 1.** $\max_{\pi_i'}J_i(\pi_i',\pi_{-i})-J_i(\pi)\le\frac{C_i}{1-\gamma}\mathbb{E}_{d^\pi}[\mathrm{gap}_i]$, where $C_i$ is a concentrability coefficient. The proof uses the performance-difference lemma.
* **Prop. 2.** A critic error of $\varepsilon_Q$ changes the gap by at most $2\varepsilon_Q$.
* **Prop. 3.** In stateless polymatrix games, $\mathcal{E}=\sum_i\mathrm{gap}_i$ exactly.
* **Prop. 4.** With $\hat Q$ detached, $\nabla_\theta\,\lambda\sum_i\mathrm{gap}_i=-\lambda\sum_i\mathbb{E}\sum_a\nabla\pi_i\hat Q_i$, which is the selfish policy gradient, so the regulariser amounts to reward mixing.

### 2.6 NashGAT-MARL objective (research hypothesis, not a validated result)
$$\mathcal{L}=L^{\text{PPO}}_{\text{team}-\mu c}-c_e\,\mathcal{H}(\pi)+\lambda\,\frac{\mathbb{E}[\mathrm{gap}]}{\mathrm{std}(\hat A)},\qquad c_i=g_i\,\frac{\mathrm{outdeg}(i)}{\max_j\mathrm{outdeg}(j)} .$$
**Limitations.** $\mathcal{E}$ is non-differentiable; the gap is a surrogate and is conservative under partial observability; $C_i$ can be large. There is no claim of convergence to a Nash equilibrium.

## Part 3 — Environment set-up
Detect the hardware, install the dependencies, try to build CityFlow, download and validate the **real** CoLight benchmark files, and set the seeds.

In [ ]:
import os, sys, subprocess, json, time, shutil, platform
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
WORK = Path("/content/nashgat_work") if IN_COLAB else Path(os.environ.get("NASHGAT_WORK", "nb_work")).resolve()
RESULTS = Path("/content/results") if IN_COLAB else WORK / "results"
DATA_ROOT = Path(os.environ.get("NASHGAT_DATA_ROOT", str(WORK)))
for p in (WORK, RESULTS):
    p.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("python", platform.python_version(), "| colab:", IN_COLAB, "| work dir:", WORK, "| results:", RESULTS)

In [ ]:
# dependencies (torch is preinstalled on Colab; PyG wheels are pure Python since 2.3)
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_geometric", "networkx", "pandas", "scipy", "matplotlib"], check=True)
import torch, numpy as np, pandas as pd
import torch_geometric
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| PyG", torch_geometric.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
# the models are tiny (12 nodes); CPU is usually as fast as GPU and avoids host-device copies per step
DEVICE = "cpu"

In [ ]:
# CityFlow: no PyPI wheel exists; build from source (needs cmake + a C++ compiler, both present on Colab)
BACKEND = "queuesim"
if PREFER_CITYFLOW:
    try:
        import cityflow  # noqa
        BACKEND = "cityflow"
    except ImportError:
        if IN_COLAB:
            r = subprocess.run("git clone --depth 1 https://github.com/cityflow-project/CityFlow.git /content/CityFlow && "
                               "pip install -q /content/CityFlow", shell=True, capture_output=True, text=True)
            print(r.stdout[-1500:], r.stderr[-1500:])
            try:
                import importlib; importlib.invalidate_caches()
                import cityflow  # noqa
                BACKEND = "cityflow"
            except ImportError as e:
                print("CityFlow build failed:", e)
        else:
            print("CityFlow not importable outside Colab in this environment.")
print("SIMULATOR BACKEND =", BACKEND,
      "" if BACKEND == "cityflow" else "  <-- QueueSim: a SEPARATE lightweight environment; do not compare its absolute numbers with CityFlow papers")

In [ ]:
# real benchmark data from the CoLight repository (sparse checkout of data/ only)
DATA_DIR = DATA_ROOT / "data/raw/colight"
if not (DATA_DIR / "data/Jinan/3_4/roadnet_3_4.json").exists():
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    cmds = [f"git clone --depth 1 --filter=blob:none --sparse https://github.com/wingsweihua/colight.git {DATA_DIR}",
            f"git -C {DATA_DIR} sparse-checkout set data/Jinan data/Hangzhou data/NewYork/28_7"]
    for c in cmds:
        subprocess.run(c, shell=True, check=True)
commit = subprocess.run(["git", "-C", str(DATA_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("CoLight data commit:", commit)

In [ ]:
# REPRODUCE_PAPER: download the released runs (trained models, logs, evaluations) from the GitHub repository
RELEASED = WORK / "released"
if MODE == "REPRODUCE_PAPER":
    if not (RELEASED / "results/traffic").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(RELEASED)], check=True)
    shutil.copytree(RELEASED / "results", RESULTS, dirs_exist_ok=True)
    runs = sorted(RESULTS.glob("traffic/*/seed*/eval.json"))
    print("released runs copied to", RESULTS, ":", len(runs), "runs, commit",
          subprocess.run(["git", "-C", str(RELEASED), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
    assert len(runs) == 52, len(runs)
else:
    print("mode", MODE, ": runs are trained in this session")

## Part 5a — Implementation: write the `nashgat` package
Each module is written to disk with `%%writefile`, so the code that runs is exactly the code shown. The modules are object-oriented and typed. Their sources are identical to the repository versions.

In [ ]:
for d in ['nashgat/games', 'nashgat/envs', 'nashgat/algos']:
    os.makedirs(d, exist_ok=True)

In [ ]:
%%writefile nashgat/__init__.py
"""NashGAT-MARL research code: graph-attention MARL with an exploitability-gap regulariser."""

__version__ = "0.1.0"

**`nashgat/utils.py`** — Seeding, device selection and JSON helpers.

In [ ]:
%%writefile nashgat/utils.py
"""Seeding, device selection and small I/O helpers."""
from __future__ import annotations

import json
import os
import random
from dataclasses import asdict, is_dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch


def set_seed(seed: int, deterministic: bool = True) -> None:
    """Seed Python, NumPy and PyTorch (CPU and CUDA)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def get_device(prefer: str = "auto") -> torch.device:
    if prefer == "cpu":
        return torch.device("cpu")
    if prefer in ("auto", "cuda") and torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def save_json(obj: Any, path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if is_dataclass(obj):
        obj = asdict(obj)
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2, default=str)

In [ ]:
%%writefile nashgat/games/__init__.py
"""Game-theoretic test beds with closed-form equilibria."""

**`nashgat/games/matrix_games.py`** — Bimatrix games with closed-form Nash equilibria and NashConv (unit-test oracles).

In [ ]:
%%writefile nashgat/games/matrix_games.py
"""Two-player bimatrix games with exact Nash equilibria and exploitability.

These are the smallest settings in which the exploitability (NashConv) used in the
NashGAT-MARL objective can be computed in closed form; they serve as unit-test
oracles for the estimators used on larger games.
"""
from __future__ import annotations

from dataclasses import dataclass
from itertools import product

import numpy as np


@dataclass(frozen=True)
class BimatrixGame:
    """Payoff matrices A (row player) and B (column player), both shape (m, n)."""

    name: str
    A: np.ndarray
    B: np.ndarray

    def payoffs(self, x: np.ndarray, y: np.ndarray) -> tuple[float, float]:
        return float(x @ self.A @ y), float(x @ self.B @ y)

    def best_response_values(self, x: np.ndarray, y: np.ndarray) -> tuple[float, float]:
        """Value of the best unilateral deviation of each player."""
        return float(np.max(self.A @ y)), float(np.max(x @ self.B))

    def nash_conv(self, x: np.ndarray, y: np.ndarray) -> float:
        """NashConv(x, y) = sum_i [max_{pi_i'} J_i(pi_i', pi_-i) - J_i(pi)] >= 0."""
        u1, u2 = self.payoffs(x, y)
        b1, b2 = self.best_response_values(x, y)
        return (b1 - u1) + (b2 - u2)

    def welfare(self, x: np.ndarray, y: np.ndarray) -> float:
        return sum(self.payoffs(x, y))


def prisoners_dilemma(T: float = 5.0, R: float = 3.0, P: float = 1.0, S: float = 0.0) -> BimatrixGame:
    """Actions (Cooperate, Defect); requires T > R > P > S."""
    assert T > R > P > S
    A = np.array([[R, S], [T, P]], dtype=float)
    return BimatrixGame("prisoners_dilemma", A, A.T.copy())


def matching_pennies() -> BimatrixGame:
    A = np.array([[1.0, -1.0], [-1.0, 1.0]])
    return BimatrixGame("matching_pennies", A, -A)


def battle_of_sexes() -> BimatrixGame:
    A = np.array([[2.0, 0.0], [0.0, 1.0]])
    B = np.array([[1.0, 0.0], [0.0, 2.0]])
    return BimatrixGame("battle_of_sexes", A, B)


def pure_nash_equilibria(game: BimatrixGame) -> list[tuple[int, int]]:
    m, n = game.A.shape
    eq = []
    for i, j in product(range(m), range(n)):
        if game.A[i, j] >= game.A[:, j].max() - 1e-12 and game.B[i, j] >= game.B[i, :].max() - 1e-12:
            eq.append((i, j))
    return eq


def mixed_nash_2x2(game: BimatrixGame) -> tuple[np.ndarray, np.ndarray] | None:
    """Fully mixed equilibrium of a 2x2 game via the indifference conditions.

    Player 2 mixes q so that player 1 is indifferent:
        q A[0,0] + (1-q) A[0,1] = q A[1,0] + (1-q) A[1,1].
    Returns None when no interior solution exists.
    """
    A, B = game.A, game.B
    den_q = A[0, 0] - A[0, 1] - A[1, 0] + A[1, 1]
    den_p = B[0, 0] - B[1, 0] - B[0, 1] + B[1, 1]
    if abs(den_q) < 1e-12 or abs(den_p) < 1e-12:
        return None
    q = (A[1, 1] - A[0, 1]) / den_q
    p = (B[1, 1] - B[1, 0]) / den_p
    if not (0.0 < p < 1.0 and 0.0 < q < 1.0):
        return None
    return np.array([p, 1.0 - p]), np.array([q, 1.0 - q])

**`nashgat/games/polymatrix.py`** — Graphical (polymatrix) games: exact exploitability and the gap / mixing / full-gradient learners.

In [ ]:
%%writefile nashgat/games/polymatrix.py
"""Graphical (polymatrix) games with exact exploitability and a differentiable
NashGAT-style objective.

Agent i chooses a_i in {0..m-1}; its payoff is
    u_i(a) = b_i[a_i] + sum_{j in N(i)} A_ij[a_i, a_j].
For independent mixed strategies x = (x_1..x_n) the action values are
    q_i(a_i) = b_i[a_i] + sum_j (A_ij x_j)[a_i]
so every quantity used by the method (J_i, best responses, NashConv, local gap)
is available in closed form. In this stateless setting the local gap equals
exploitability exactly (Proposition 1 in MATHEMATICAL_MODEL.md with C_i = 1).
"""
from __future__ import annotations

from dataclasses import dataclass, field
from itertools import product

import numpy as np
import torch


@dataclass
class PolymatrixGame:
    n_agents: int
    n_actions: int
    edges: list[tuple[int, int]]  # undirected
    pair_payoff: dict[tuple[int, int], np.ndarray] = field(default_factory=dict)  # (i,j) -> (m,m) payoff of i
    bias: np.ndarray | None = None  # (n, m)

    def neighbours(self, i: int) -> list[int]:
        return [j for (a, b) in self.edges for (u, j) in ((a, b), (b, a)) if u == i]

    def action_values(self, x: np.ndarray) -> np.ndarray:
        """q[i, a] for mixed profile x of shape (n, m)."""
        q = np.zeros((self.n_agents, self.n_actions)) if self.bias is None else self.bias.copy()
        for (i, j), M in self.pair_payoff.items():
            q[i] += M @ x[j]
        return q

    def payoffs(self, x: np.ndarray) -> np.ndarray:
        return (self.action_values(x) * x).sum(1)

    def nash_conv(self, x: np.ndarray) -> float:
        q = self.action_values(x)
        return float((q.max(1) - (q * x).sum(1)).sum())

    def welfare(self, x: np.ndarray) -> float:
        return float(self.payoffs(x).sum())

    def pure_nash_equilibria(self, max_profiles: int = 2**16) -> list[tuple[int, ...]]:
        """Brute-force enumeration (only for small games)."""
        assert self.n_actions**self.n_agents <= max_profiles, "game too large for enumeration"
        eq = []
        eye = np.eye(self.n_actions)
        for prof in product(range(self.n_actions), repeat=self.n_agents):
            x = eye[list(prof)]
            if self.nash_conv(x) < 1e-9:
                eq.append(prof)
        return eq

    # ---- torch versions used by the gradient-based learners ----
    def dense_payoff(self) -> np.ndarray:
        """(n, n, m, m) tensor with M[i, j] = A_ij for neighbours and 0 otherwise."""
        M = np.zeros((self.n_agents, self.n_agents, self.n_actions, self.n_actions))
        for (i, j), P in self.pair_payoff.items():
            M[i, j] = P
        return M

    def torch_action_values(self, x: torch.Tensor) -> torch.Tensor:
        if not hasattr(self, "_dense"):
            self._dense = torch.as_tensor(self.dense_payoff())
        q = torch.einsum("ijab,jb->ia", self._dense.to(x.dtype), x)
        if self.bias is not None:
            q = q + torch.as_tensor(self.bias, dtype=x.dtype)
        return q


def networked_prisoners_dilemma(rows: int, cols: int, T: float = 5.0, R: float = 3.0,
                                P: float = 1.0, S: float = 0.0) -> PolymatrixGame:
    """Pairwise PD on a grid. Defect (action 1) is strictly dominant, so the
    unique Nash equilibrium is all-defect, while all-cooperate maximises welfare."""
    n = rows * cols
    edges = []
    for r in range(rows):
        for c in range(cols):
            i = r * cols + c
            if c + 1 < cols:
                edges.append((i, i + 1))
            if r + 1 < rows:
                edges.append((i, i + cols))
    M = np.array([[R, S], [T, P]], dtype=float)
    pp = {}
    for i, j in edges:
        pp[(i, j)] = M.copy()
        pp[(j, i)] = M.copy()
    return PolymatrixGame(n, 2, edges, pp)


def grid_anti_coordination(rows: int, cols: int, gain: float = 1.0) -> PolymatrixGame:
    """Agents gain when they pick a different action from each neighbour
    (two-colouring). On a bipartite grid the two checkerboard profiles are pure Nash
    equilibria and also welfare-optimal; used to check that the regulariser does not
    move learners away from an equilibrium that is already efficient."""
    base = networked_prisoners_dilemma(rows, cols)
    M = np.array([[0.0, gain], [gain, 0.0]])
    pp = {k: M.copy() for k in base.pair_payoff}
    return PolymatrixGame(base.n_agents, 2, base.edges, pp)


def train_softmax_profile(game: PolymatrixGame, objective: str, lam: float = 0.0, steps: int = 3000,
                          lr: float = 0.1, seed: int = 0, init_scale: float = 0.1) -> dict:
    """Exact-gradient learning of independent softmax policies.

    objective:
      'independent' : each agent ascends its own payoff (simultaneous gradient play)
      'welfare'     : ascend sum_i J_i
      'welfare_gap' : ascend sum_i J_i - lam * sum_i gap_i, where
                      gap_i = max_a q_i(a) - x_i . q_i with q_i treated as fixed
                      (stop-gradient through the other agents), mirroring the
                      critic-based regulariser used in the traffic experiments.
      'welfare_mix' : ascend sum_i J_i + lam * sum_i x_i . q_i(fixed); its gradient is
                      identical to 'welfare_gap' (Proposition 4).
      'welfare_fullgap': ascend sum_i J_i - lam * NashConv(x) with the exact gradient.
    """
    g = torch.Generator().manual_seed(seed)
    logits = (init_scale * torch.randn(game.n_agents, game.n_actions, generator=g, dtype=torch.float64)).requires_grad_()
    opt = torch.optim.Adam([logits], lr=lr)
    hist = []
    for t in range(steps):
        x = torch.softmax(logits, -1)
        if objective == "independent":
            # each agent differentiates only its own payoff w.r.t. its own policy
            q = game.torch_action_values(x.detach())
            loss = -(x * q).sum()
        elif objective in ("welfare", "welfare_gap", "welfare_mix", "welfare_fullgap"):
            q = game.torch_action_values(x)
            loss = -(x * q).sum()
            if objective == "welfare_gap":
                # semi-gradient: Q_i treated as fixed, as with a detached critic
                qd = game.torch_action_values(x.detach()).detach()
                gap = qd.max(1).values - (x * qd).sum(1)
                loss = loss + lam * gap.sum()
            elif objective == "welfare_mix":
                # reward mixing: welfare + lam * each agent's own payoff (independent gradient)
                qd = game.torch_action_values(x.detach()).detach()
                loss = loss - lam * (x * qd).sum()
            elif objective == "welfare_fullgap":
                # full gradient of NashConv: also moves the *other* agents so as to remove
                # agent i's incentive to deviate (not available with a detached critic)
                gap = q.max(1).values - (x * q).sum(1)
                loss = loss + lam * gap.sum()
        else:
            raise ValueError(objective)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if t % max(1, steps // 100) == 0 or t == steps - 1:
            xn = x.detach().numpy()
            hist.append((t, game.welfare(xn), game.nash_conv(xn)))
    xn = torch.softmax(logits, -1).detach().numpy()
    return {"x": xn, "welfare": game.welfare(xn), "nash_conv": game.nash_conv(xn), "history": hist}

In [ ]:
%%writefile nashgat/envs/__init__.py
"""Traffic environments (QueueSim and CityFlow backends)."""

**`nashgat/envs/roadnet.py`** — CityFlow roadnet/flow parser and validator.

In [ ]:
%%writefile nashgat/envs/roadnet.py
"""Parsing and validation of CityFlow road-network and flow files."""
from __future__ import annotations

import json
import math
from dataclasses import dataclass, field
from pathlib import Path


@dataclass
class Road:
    id: str
    start: str
    end: str
    length: float
    n_lanes: int
    max_speed: float


@dataclass
class Movement:
    """One CityFlow roadLink of an intersection (startRoad -> endRoad)."""

    index: int
    kind: str
    start_road: str
    end_road: str
    n_lanes: int


@dataclass
class Intersection:
    id: str
    virtual: bool
    roads: list[str]
    movements: list[Movement] = field(default_factory=list)
    phases: list[list[int]] = field(default_factory=list)  # availableRoadLinks per light phase
    phase_times: list[int] = field(default_factory=list)


@dataclass
class RoadNet:
    roads: dict[str, Road]
    intersections: dict[str, Intersection]

    @property
    def agents(self) -> list[str]:
        """Signalised (non-virtual) intersections, sorted for a stable agent order."""
        return sorted(i.id for i in self.intersections.values() if not i.virtual)

    def neighbour_edges(self) -> list[tuple[int, int, str]]:
        """Directed agent edges (j -> i) induced by roads joining two signalised
        intersections, with the connecting road id."""
        idx = {a: k for k, a in enumerate(self.agents)}
        out = []
        for r in self.roads.values():
            if r.start in idx and r.end in idx:
                out.append((idx[r.start], idx[r.end], r.id))
        return sorted(out)


def _length(points: list[dict]) -> float:
    return sum(math.dist((points[k]["x"], points[k]["y"]), (points[k + 1]["x"], points[k + 1]["y"]))
               for k in range(len(points) - 1))


def load_roadnet(path: str | Path) -> RoadNet:
    with open(path, encoding="utf-8") as fh:
        raw = json.load(fh)
    roads = {}
    for r in raw["roads"]:
        roads[r["id"]] = Road(r["id"], r["startIntersection"], r["endIntersection"], _length(r["points"]),
                              len(r["lanes"]), float(r["lanes"][0]["maxSpeed"]))
    inters = {}
    for it in raw["intersections"]:
        inter = Intersection(it["id"], bool(it["virtual"]), list(it.get("roads", [])))
        for k, rl in enumerate(it.get("roadLinks", [])):
            lanes = {ll["startLaneIndex"] for ll in rl["laneLinks"]}
            inter.movements.append(Movement(k, rl["type"], rl["startRoad"], rl["endRoad"], max(1, len(lanes))))
        if not inter.virtual:
            tl = it["trafficLight"]
            inter.phases = [list(p["availableRoadLinks"]) for p in tl["lightphases"]]
            inter.phase_times = [int(p["time"]) for p in tl["lightphases"]]
        inters[it["id"]] = inter
    return RoadNet(roads, inters)


def load_flows(path: str | Path) -> list[dict]:
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)


def expand_departures(flows: list[dict]) -> list[tuple[int, list[str]]]:
    """Expand CityFlow flow entries into (departure_time, route) per vehicle."""
    out = []
    for f in flows:
        t, end, iv = float(f["startTime"]), float(f["endTime"]), float(f.get("interval", 1.0))
        iv = max(iv, 1e-9)
        while t <= end + 1e-9:
            out.append((int(round(t)), list(f["route"])))
            if end == f["startTime"]:
                break
            t += iv
    out.sort(key=lambda z: z[0])
    return out


def validate(net: RoadNet, flows: list[dict]) -> dict:
    """Consistency checks reported in DATASET_CARD.md."""
    issues = []
    for r in net.roads.values():
        if r.start not in net.intersections or r.end not in net.intersections:
            issues.append(f"road {r.id} references unknown intersection")
        if r.length <= 0:
            issues.append(f"road {r.id} non-positive length")
    for it in net.intersections.values():
        for m in it.movements:
            if m.start_road not in net.roads or m.end_road not in net.roads:
                issues.append(f"{it.id}: movement {m.index} references unknown road")
            elif net.roads[m.start_road].end != it.id or net.roads[m.end_road].start != it.id:
                issues.append(f"{it.id}: movement {m.index} not incident to intersection")
        for p in it.phases:
            if any(k >= len(it.movements) for k in p):
                issues.append(f"{it.id}: phase references missing roadLink")
    bad_routes = 0
    for f in flows:
        rt = f["route"]
        if any(r not in net.roads for r in rt):
            bad_routes += 1
            continue
        for a, b in zip(rt[:-1], rt[1:]):
            if net.roads[a].end != net.roads[b].start:
                bad_routes += 1
                break
    deps = expand_departures(flows)
    return {
        "n_intersections": len(net.intersections),
        "n_signalised": len(net.agents),
        "n_virtual": sum(i.virtual for i in net.intersections.values()),
        "n_roads": len(net.roads),
        "n_phases": sorted({len(net.intersections[a].phases) for a in net.agents}),
        "n_movements": sorted({len(net.intersections[a].movements) for a in net.agents}),
        "road_length_range_m": (min(r.length for r in net.roads.values()), max(r.length for r in net.roads.values())),
        "n_flow_entries": len(flows),
        "n_vehicles": len(deps),
        "departure_range_s": (deps[0][0], deps[-1][0]) if deps else None,
        "route_length_hist": _hist([len(f["route"]) for f in flows]),
        "invalid_routes": bad_routes,
        "issues": issues,
    }


def _hist(xs: list[int]) -> dict[int, int]:
    h: dict[int, int] = {}
    for x in xs:
        h[x] = h.get(x, 0) + 1
    return dict(sorted(h.items()))

**`nashgat/envs/queue_sim.py`** — **QueueSim**: the lightweight, *separate* mesoscopic simulator used when CityFlow is unavailable.

In [ ]:
%%writefile nashgat/envs/queue_sim.py
"""Lightweight mesoscopic queue simulator for CityFlow road networks ("QueueSim").

This is NOT CityFlow. It is a separate, deliberately simple experimental environment
used where CityFlow cannot be built. It reads the original CityFlow roadnet/flow
files and models:

* free-flow travel on each road (time = length / maxSpeed, rounded up),
* point queues per movement (CityFlow roadLink) at the downstream intersection,
* saturation-flow discharge (one vehicle per `sat_headway` s per lane) during green,
* finite road storage (length * lanes / vehicle spacing) causing spill-back,
* a yellow interval on every phase change during which only the links of
  light phase 0 (right turns in the CoLight networks) may discharge.

There is no car-following, lane changing or vehicle-level kinematics, so absolute
travel times are not comparable with CityFlow numbers published elsewhere.
"""
from __future__ import annotations

import math
from collections import deque
from dataclasses import dataclass

import numpy as np

from .roadnet import RoadNet


@dataclass
class SimConfig:
    yellow_time: int = 3
    sat_headway: float = 2.0  # s per vehicle per lane
    vehicle_spacing: float = 7.5  # m of storage per queued vehicle (length 5 + min gap 2.5)
    horizon: int = 3600


class QueueSim:
    def __init__(self, net: RoadNet, departures: list[tuple[int, list[str]]], cfg: SimConfig | None = None):
        self.net = net
        self.cfg = cfg or SimConfig()
        self.departures = departures
        self.agents = net.agents
        self.agent_index = {a: k for k, a in enumerate(self.agents)}
        self.road_ids = sorted(net.roads)
        self.ridx = {r: k for k, r in enumerate(self.road_ids)}
        R = len(self.road_ids)
        self.fft = np.array([max(1, math.ceil(net.roads[r].length / net.roads[r].max_speed)) for r in self.road_ids])
        self.cap = np.array([max(1, int(net.roads[r].length * net.roads[r].n_lanes // self.cfg.vehicle_spacing))
                             for r in self.road_ids])
        self.road_lanes = np.array([net.roads[r].n_lanes for r in self.road_ids])
        # movements: global index per (agent, link)
        self.mov_key: list[tuple[int, int]] = []  # (from road idx, to road idx)
        self.mov_lanes: list[int] = []
        self.mov_agent: list[int] = []
        self.agent_movs: list[list[int]] = []
        self.agent_phases: list[list[list[int]]] = []  # phases as global movement ids
        self.agent_yellow_links: list[list[int]] = []
        key_to_mov: dict[tuple[int, int], int] = {}
        for a in self.agents:
            it = net.intersections[a]
            ids = []
            for m in it.movements:
                g = len(self.mov_key)
                k = (self.ridx[m.start_road], self.ridx[m.end_road])
                self.mov_key.append(k)
                self.mov_lanes.append(m.n_lanes)
                self.mov_agent.append(self.agent_index[a])
                key_to_mov[k] = g
                ids.append(g)
            self.agent_movs.append(ids)
            self.agent_phases.append([[ids[k] for k in p] for p in it.phases])
            self.agent_yellow_links.append([ids[k] for k in it.phases[0]] if it.phases else [])
        self.key_to_mov = key_to_mov
        self.n_phases = [len(p) - 1 for p in self.agent_phases]  # controllable phases exclude phase 0
        self.n_mov = len(self.mov_key)
        self.mov_rate = np.array(self.mov_lanes, dtype=float) / self.cfg.sat_headway
        self.failed_agents: set[int] = set()
        self.reset()

    # ------------------------------------------------------------------ state
    def reset(self) -> None:
        R = len(self.road_ids)
        self.t = 0
        self.count = np.zeros(R, dtype=np.int64)  # vehicles on road (moving or queued)
        self.transit: list[deque] = [deque() for _ in range(R)]
        self.queues: list[deque] = [deque() for _ in range(self.n_mov)]
        self.approach = np.zeros(self.n_mov, dtype=np.int64)  # moving vehicles heading to each movement
        self.credit = np.zeros(self.n_mov)
        self.entry: dict[int, deque] = {}
        self.entry_credit = np.zeros(R)
        self.free_queues: dict[tuple[int, int], deque] = {}  # transitions with no roadLink (should not occur)
        n_veh = len(self.departures)
        self.route = [[self.ridx[r] for r in rt] for _, rt in self.departures]
        self.depart = np.array([d for d, _ in self.departures], dtype=np.int64)
        self.pos = np.zeros(n_veh, dtype=np.int64)
        self.finish = np.full(n_veh, -1, dtype=np.int64)
        self.wait = np.zeros(n_veh, dtype=np.int64)
        self.queue_join = np.zeros(n_veh, dtype=np.int64)
        self.next_dep = 0
        nA = len(self.agents)
        self.phase = np.ones(nA, dtype=np.int64)  # light phase index (1..P)
        self.yellow_left = np.zeros(nA, dtype=np.int64)
        self.pending_phase = np.ones(nA, dtype=np.int64)
        self.n_missing_links = 0
        self.green_time = np.zeros(nA, dtype=np.int64)
        self._fixed_timer = np.zeros(nA, dtype=np.int64)

    # --------------------------------------------------------------- control
    def set_phases(self, phases: np.ndarray) -> None:
        """phases: controllable phase per agent, 0-based (maps to light phase 1..P)."""
        for a, p in enumerate(phases):
            if a in self.failed_agents:
                continue  # failed controller follows its fixed-time fallback in _fixed_time_step
            lp = int(p) + 1
            if lp != self.phase[a] and self.yellow_left[a] == 0:
                self.yellow_left[a] = self.cfg.yellow_time
                self.pending_phase[a] = lp
            elif self.yellow_left[a] > 0:
                self.pending_phase[a] = lp

    def _fixed_time_step(self) -> None:
        for a in self.failed_agents:
            self._fixed_timer[a] += 1
            if self._fixed_timer[a] >= 30 and self.yellow_left[a] == 0:
                self._fixed_timer[a] = 0
                self.yellow_left[a] = self.cfg.yellow_time
                self.pending_phase[a] = (self.phase[a] % self.n_phases[a]) + 1

    # ------------------------------------------------------------- dynamics
    def _enter_road(self, v: int, r: int) -> None:
        self.count[r] += 1
        self.transit[r].append((self.t + int(self.fft[r]), v))
        rt = self.route[v]
        p = self.pos[v]
        if p + 1 < len(rt):
            m = self.key_to_mov.get((r, rt[p + 1]))
            if m is not None:
                self.approach[m] += 1

    def step_second(self) -> None:
        t = self.t
        # 1) new departures join the entry buffer of their first road
        while self.next_dep < len(self.depart) and self.depart[self.next_dep] <= t:
            v = self.next_dep
            self.entry.setdefault(self.route[v][0], deque()).append(v)
            self.next_dep += 1
        # 2) entry into the network, limited by road storage and lane capacity
        for r, buf in self.entry.items():
            if not buf:
                continue
            self.entry_credit[r] = min(self.entry_credit[r] + self.road_lanes[r] / self.cfg.sat_headway, self.road_lanes[r])
            while buf and self.entry_credit[r] >= 1.0 and self.count[r] < self.cap[r]:
                v = buf.popleft()
                self.wait[v] += t - self.depart[v]
                self._enter_road(v, r)
                self.entry_credit[r] -= 1.0
        # 3) vehicles reaching the end of a road join a movement queue or leave
        for r in range(len(self.transit)):
            tr = self.transit[r]
            while tr and tr[0][0] <= t:
                _, v = tr.popleft()
                rt = self.route[v]
                p = self.pos[v]
                if p + 1 >= len(rt):
                    self.count[r] -= 1
                    self.finish[v] = t
                    continue
                m = self.key_to_mov.get((r, rt[p + 1]))
                if m is None:
                    self.n_missing_links += 1
                    self.free_queues.setdefault((r, rt[p + 1]), deque()).append(v)
                    continue
                self.approach[m] -= 1
                self.queues[m].append(v)
                self.queue_join[v] = t
        # 4) discharge on green movements
        self._fixed_time_step()
        for a in range(len(self.agents)):
            if self.yellow_left[a] > 0:
                active = self.agent_yellow_links[a]
                self.yellow_left[a] -= 1
                if self.yellow_left[a] == 0:
                    self.phase[a] = self.pending_phase[a]
                    self.green_time[a] = 0
            else:
                active = self.agent_phases[a][self.phase[a]]
                self.green_time[a] += 1
            for m in active:
                q = self.queues[m]
                if not q:
                    self.credit[m] = 0.0
                    continue
                self.credit[m] = min(self.credit[m] + self.mov_rate[m], self.mov_lanes[m])
                r_in, r_out = self.mov_key[m]
                while q and self.credit[m] >= 1.0 and self.count[r_out] < self.cap[r_out]:
                    v = q.popleft()
                    self.wait[v] += t - self.queue_join[v]
                    self.count[r_in] -= 1
                    self.pos[v] += 1
                    self._enter_road(v, r_out)
                    self.credit[m] -= 1.0
        for (r_in, r_out), q in self.free_queues.items():  # unsignalised fallback
            while q and self.count[r_out] < self.cap[r_out]:
                v = q.popleft()
                self.count[r_in] -= 1
                self.pos[v] += 1
                self._enter_road(v, r_out)
        self.t += 1

    # ------------------------------------------------------------ read-outs
    def queue_lengths(self) -> np.ndarray:
        return np.fromiter((len(q) for q in self.queues), dtype=np.int64, count=self.n_mov)

    def metrics(self) -> dict:
        """CityFlow-style aggregate metrics at the current time.

        Travel time counts from the scheduled departure; vehicles still in the
        network (or waiting to enter) contribute their elapsed time so far.
        """
        started = self.depart[: self.next_dep]
        fin = self.finish[: self.next_dep]
        tt = np.where(fin >= 0, fin - started, self.t - started)
        q = self.queue_lengths()
        # waiting time of vehicles currently queued or waiting at entry is added on top of the accrued wait
        w = self.wait[: self.next_dep].astype(float).copy()
        for qq in self.queues:
            for v in qq:
                w[v] += self.t - self.queue_join[v]
        for buf in self.entry.values():
            for v in buf:
                w[v] += self.t - self.depart[v]
        return {
            "time": self.t,
            "vehicles_departed": int(self.next_dep),
            "throughput": int((fin >= 0).sum()),
            "avg_travel_time": float(tt.mean()) if len(tt) else 0.0,
            "avg_waiting_time": float(w.mean()) if len(w) else 0.0,
            "total_queue": int(q.sum()),
            "entry_backlog": int(sum(len(b) for b in self.entry.values())),
        }

**`nashgat/envs/cityflow_backend.py`** — CityFlow backend with the same interface (used when `import cityflow` succeeds).

In [ ]:
%%writefile nashgat/envs/cityflow_backend.py
"""CityFlow backend exposing the same interface as QueueSim.

Requires the `cityflow` Python package (built from https://github.com/cityflow-project/CityFlow).
It could not be built in the local development environment (Windows, no C++ toolchain),
so this module is exercised only in the Colab notebook. Queue = number of vehicles
with speed < 0.1 m/s on the movement's start lane(s), as in CoLight/PressLight.
"""
from __future__ import annotations

import json
import os
import tempfile
from pathlib import Path

import numpy as np

from .roadnet import RoadNet


class CityFlowBackend:
    def __init__(self, net: RoadNet, cfg):
        import cityflow  # noqa: F401  (ImportError is the signal that the backend is unavailable)

        self.net, self.cfg = net, cfg
        self.agents = net.agents
        self.road_ids = sorted(net.roads)
        self.ridx = {r: k for k, r in enumerate(self.road_ids)}
        self.cap = np.array([max(1, int(net.roads[r].length * net.roads[r].n_lanes // 7.5)) for r in self.road_ids])
        self.mov_key, self.mov_lanes, self.agent_movs, self.agent_phases = [], [], [], []
        self.start_lanes: list[list[str]] = []
        raw = json.load(open(cfg.roadnet, encoding="utf-8"))
        rl_lanes = {}
        for it in raw["intersections"]:
            for k, rl in enumerate(it.get("roadLinks", [])):
                rl_lanes[(it["id"], k)] = sorted({f"{rl['startRoad']}_{ll['startLaneIndex']}" for ll in rl["laneLinks"]})
        for a in self.agents:
            it = net.intersections[a]
            ids = []
            for m in it.movements:
                ids.append(len(self.mov_key))
                self.mov_key.append((self.ridx[m.start_road], self.ridx[m.end_road]))
                self.mov_lanes.append(m.n_lanes)
                self.start_lanes.append(rl_lanes[(a, m.index)])
            self.agent_movs.append(ids)
            self.agent_phases.append([[ids[k] for k in p] for p in it.phases])
        self.n_mov = len(self.mov_key)
        self.n_phases = [len(p) - 1 for p in self.agent_phases]
        self.failed_agents: set[int] = set()
        self._tmp = tempfile.mkdtemp(prefix="cityflow_")
        conf = {"interval": 1.0, "seed": int(cfg.seed), "dir": "", "roadnetFile": os.path.abspath(cfg.roadnet),
                "flowFile": os.path.abspath(cfg.flow), "rlTrafficLight": True, "saveReplay": False,
                "roadnetLogFile": "", "replayLogFile": "", "laneChange": False}
        self._conf_path = Path(self._tmp) / "config.json"
        self._conf_path.write_text(json.dumps(conf))
        self.reset()

    def reset(self) -> None:
        import cityflow

        self.eng = cityflow.Engine(str(self._conf_path), thread_num=1)
        nA = len(self.agents)
        self.t = 0
        self.phase = np.ones(nA, dtype=np.int64)
        self.pending_phase = np.ones(nA, dtype=np.int64)
        self.yellow_left = np.zeros(nA, dtype=np.int64)
        self._fixed_timer = np.zeros(nA, dtype=np.int64)
        self._wait_accum = 0.0
        self._prev_vehicles: set[str] = set()
        self._finished = 0
        for a, aid in enumerate(self.agents):
            self.eng.set_tl_phase(aid, 1)
        self._refresh()

    def set_phases(self, phases: np.ndarray) -> None:
        for a, p in enumerate(phases):
            if a in self.failed_agents:
                continue
            lp = int(p) + 1
            if lp != self.phase[a] and self.yellow_left[a] == 0:
                self.yellow_left[a] = self.cfg.yellow_time
                self.pending_phase[a] = lp
                self.eng.set_tl_phase(self.agents[a], 0)
            elif self.yellow_left[a] > 0:
                self.pending_phase[a] = lp

    def step_second(self) -> None:
        for a in self.failed_agents:
            self._fixed_timer[a] += 1
            if self._fixed_timer[a] >= 30 and self.yellow_left[a] == 0:
                self._fixed_timer[a] = 0
                self.yellow_left[a] = self.cfg.yellow_time
                self.pending_phase[a] = (self.phase[a] % self.n_phases[a]) + 1
                self.eng.set_tl_phase(self.agents[a], 0)
        self.eng.next_step()
        self.t += 1
        for a in range(len(self.agents)):
            if self.yellow_left[a] > 0:
                self.yellow_left[a] -= 1
                if self.yellow_left[a] == 0:
                    self.phase[a] = self.pending_phase[a]
                    self.eng.set_tl_phase(self.agents[a], int(self.phase[a]))
        self._refresh()
        self._wait_accum += float(sum(self._lane_wait.values()))

    def _refresh(self) -> None:
        self._lane_wait = self.eng.get_lane_waiting_vehicle_count()
        self._lane_cnt = self.eng.get_lane_vehicle_count()
        cur = set(self.eng.get_vehicles(include_waiting=True))
        self._finished += len(self._prev_vehicles - cur)  # vehicles that left the network this second
        self._prev_vehicles = cur
        self.count = np.zeros(len(self.road_ids), dtype=np.int64)
        for lane, c in self._lane_cnt.items():
            road = lane.rsplit("_", 1)[0]
            if road in self.ridx:
                self.count[self.ridx[road]] += c
        q = np.zeros(self.n_mov, dtype=np.int64)
        app = np.zeros(self.n_mov, dtype=np.int64)
        for m, lanes in enumerate(self.start_lanes):
            q[m] = sum(self._lane_wait.get(l, 0) for l in lanes)
            app[m] = sum(self._lane_cnt.get(l, 0) for l in lanes) - q[m]
        self._q = q
        self.approach = app

    def queue_lengths(self) -> np.ndarray:
        return self._q

    def metrics(self) -> dict:
        return {
            "time": self.t,
            "avg_travel_time": float(self.eng.get_average_travel_time()),
            "vehicles_in_network": int(self.eng.get_vehicle_count()),
            "avg_waiting_time": float("nan"),  # not exposed per vehicle by CityFlow; see total_waiting_vehicle_seconds
            "total_waiting_vehicle_seconds": self._wait_accum,
            "total_queue": int(self._q.sum()),
            "throughput": int(self._finished),
        }

**`nashgat/envs/traffic_env.py`** — Multi-agent environment: observations, individual rewards, edge features, robustness options.

In [ ]:
%%writefile nashgat/envs/traffic_env.py
"""Multi-agent traffic-signal environment (one agent per signalised intersection).

The environment is a partially observable general-sum Markov game:
  * agent i observes the queues/approaching counts of its own 12 movements and its
    current phase,
  * action a_i selects one of the controllable light phases for the next
    `decision_interval` seconds (a yellow interval is inserted on change),
  * reward r_i = -(sum of queued vehicles on i's movements) / reward_scale,
    measured at the end of the decision interval.

Rewards are individual: an intersection that flushes its own queues into a
congested neighbour is rewarded even though the neighbour pays the cost, which
is what makes the game general-sum rather than fully cooperative.

The same API is served by two backends: QueueSim (always available) and CityFlow
(when the `cityflow` package can be imported). Results from the two must never be
pooled.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path

import numpy as np

from .queue_sim import QueueSim, SimConfig
from .roadnet import RoadNet, expand_departures, load_flows, load_roadnet


@dataclass
class EnvConfig:
    roadnet: str
    flow: str
    backend: str = "queuesim"  # 'queuesim' | 'cityflow'
    horizon: int = 3600
    decision_interval: int = 10
    yellow_time: int = 3
    reward_scale: float = 20.0
    obs_scale: float = 20.0
    obs_noise_std: float = 0.0
    demand_scale: float = 1.0  # >1 duplicates, <1 thins vehicles (transformed demand)
    departure_jitter: int = 0  # +- seconds of uniform jitter applied to departures
    failed_agents: list[int] = field(default_factory=list)
    seed: int = 0


def transform_departures(deps: list[tuple[int, list[str]]], scale: float, jitter: int,
                         rng: np.random.Generator, horizon: int) -> list[tuple[int, list[str]]]:
    """Seeded demand perturbation used only for robustness experiments."""
    out = []
    for t, rt in deps:
        k = int(scale) + (1 if rng.random() < scale - int(scale) else 0)
        for _ in range(k):
            tt = t + (int(rng.integers(-jitter, jitter + 1)) if jitter > 0 else 0)
            out.append((int(np.clip(tt, 0, horizon - 1)), rt))
    out.sort(key=lambda z: z[0])
    return out


class TrafficMAEnv:
    def __init__(self, cfg: EnvConfig):
        self.cfg = cfg
        self.net: RoadNet = load_roadnet(cfg.roadnet)
        self.raw_flows = load_flows(cfg.flow)
        self.agents = self.net.agents
        self.n_agents = len(self.agents)
        self.rng = np.random.default_rng(cfg.seed)
        self._build_backend()
        nm = {len(self.net.intersections[a].movements) for a in self.agents}
        npz = {len(self.net.intersections[a].phases) - 1 for a in self.agents}
        assert len(nm) == 1 and len(npz) == 1, "heterogeneous intersections are not supported"
        self.n_movements = nm.pop()
        self.n_actions = npz.pop()
        self.obs_dim = 2 * self.n_movements + self.n_actions
        # directed agent graph (j -> i) from connecting roads
        e = self.net.neighbour_edges()
        self.edge_index = np.array([[s for s, _, _ in e], [d for _, d, _ in e]], dtype=np.int64).reshape(2, -1)
        self.edge_roads = [r for _, _, r in e]
        self.edge_dim = 2
        self._edge_road_idx = [self.sim.ridx[r] for r in self.edge_roads]
        self._edge_movs = [[m for m, (a, _) in enumerate(self.sim.mov_key) if a == ri] for ri in self._edge_road_idx]

    def _departures(self) -> list[tuple[int, list[str]]]:
        deps = expand_departures(self.raw_flows)
        if self.cfg.demand_scale != 1.0 or self.cfg.departure_jitter > 0:
            deps = transform_departures(deps, self.cfg.demand_scale, self.cfg.departure_jitter, self.rng,
                                        self.cfg.horizon)
        return [d for d in deps if d[0] < self.cfg.horizon]

    def _build_backend(self) -> None:
        if self.cfg.backend == "queuesim":
            self.sim = QueueSim(self.net, self._departures(),
                                SimConfig(yellow_time=self.cfg.yellow_time, horizon=self.cfg.horizon))
        elif self.cfg.backend == "cityflow":
            from .cityflow_backend import CityFlowBackend

            self.sim = CityFlowBackend(self.net, self.cfg)
        else:
            raise ValueError(self.cfg.backend)
        self.sim.failed_agents = set(self.cfg.failed_agents)

    # ------------------------------------------------------------------ API
    def reset(self, seed: int | None = None) -> tuple[np.ndarray, dict]:
        if seed is not None:
            self.rng = np.random.default_rng(seed)
            if self.cfg.demand_scale != 1.0 or self.cfg.departure_jitter > 0:
                self._build_backend()
        self.sim.reset()
        self.steps = 0
        return self._obs(), {}

    def step(self, actions: np.ndarray) -> tuple[np.ndarray, np.ndarray, bool, bool, dict]:
        actions = np.asarray(actions, dtype=np.int64)
        assert actions.shape == (self.n_agents,)
        assert actions.min() >= 0 and actions.max() < self.n_actions
        self.sim.set_phases(actions)
        for _ in range(self.cfg.decision_interval):
            self.sim.step_second()
        self.steps += 1
        q = self.sim.queue_lengths()
        rewards = -np.array([q[m].sum() for m in self.sim.agent_movs], dtype=np.float64) / self.cfg.reward_scale
        truncated = self.sim.t >= self.cfg.horizon
        info = {"queue_per_agent": -rewards * self.cfg.reward_scale}
        if truncated:
            info["metrics"] = self.sim.metrics()
        return self._obs(), rewards, False, truncated, info

    def _obs(self) -> np.ndarray:
        q = self.sim.queue_lengths()
        app = self.sim.approach
        obs = np.zeros((self.n_agents, self.obs_dim), dtype=np.float32)
        for a, movs in enumerate(self.sim.agent_movs):
            obs[a, : self.n_movements] = q[movs] / self.cfg.obs_scale
            obs[a, self.n_movements: 2 * self.n_movements] = app[movs] / self.cfg.obs_scale
            obs[a, 2 * self.n_movements + int(self.sim.phase[a]) - 1] = 1.0
        if self.cfg.obs_noise_std > 0:
            obs[:, : 2 * self.n_movements] += self.rng.normal(0, self.cfg.obs_noise_std,
                                                              (self.n_agents, 2 * self.n_movements)).astype(np.float32)
        return obs

    def edge_features(self) -> np.ndarray:
        """Traffic coupling on each directed agent edge j->i: occupancy of the
        connecting road and the share of it already queued at i."""
        feats = np.zeros((len(self.edge_roads), self.edge_dim), dtype=np.float32)
        q = self.sim.queue_lengths()
        for k, ri in enumerate(self._edge_road_idx):
            feats[k, 0] = self.sim.count[ri] / self.sim.cap[ri]
            feats[k, 1] = q[self._edge_movs[k]].sum() / self.cfg.obs_scale
        return feats

    def metrics(self) -> dict:
        return self.sim.metrics()


def default_env_config(root: str | Path, city: str = "jinan", flow_variant: str = "real", **kw) -> EnvConfig:
    root = Path(root)
    if city == "jinan":
        d = root / "data/raw/colight/data/Jinan/3_4"
        rn = d / "roadnet_3_4.json"
        fl = d / {"real": "anon_3_4_jinan_real.json", "2000": "anon_3_4_jinan_real_2000.json",
                  "2500": "anon_3_4_jinan_real_2500.json"}[flow_variant]
    elif city == "hangzhou":
        d = root / "data/raw/colight/data/Hangzhou/4_4"
        rn = d / "roadnet_4_4.json"
        fl = d / {"real": "anon_4_4_hangzhou_real.json", "5734": "anon_4_4_hangzhou_real_5734.json",
                  "5816": "anon_4_4_hangzhou_real_5816.json"}[flow_variant]
    elif city == "newyork":
        d = root / "data/raw/colight/data/NewYork/28_7"
        rn = d / "roadnet_28_7.json"
        fl = d / {"double": "anon_28_7_newyork_real_double.json", "triple": "anon_28_7_newyork_real_triple.json",
                  "real": "anon_28_7_newyork_real_double.json"}[flow_variant]
    else:
        raise ValueError(city)
    return EnvConfig(roadnet=str(rn), flow=str(fl), **kw)

**`nashgat/graph.py`** — Dynamic graph $G_t=(V,E_t)$ with communication gates and link failures; graph batching.

In [ ]:
%%writefile nashgat/graph.py
"""Dynamic communication graph G_t = (V, E_t) and graph batching utilities."""
from __future__ import annotations

import numpy as np
import torch


class DynamicGraphBuilder:
    """Maintains the active edge set E_t over the static road topology E_0.

    An edge j->i of E_0 is active at step t iff sender j chose to transmit at t-1
    (communication gate g_j = 1) and the link did not fail. With gating disabled all
    edges are active (static graph). Self-loops are added inside the GNN layers and
    never count as communication.
    """

    def __init__(self, edge_index: np.ndarray, n_agents: int, gated: bool, fail_prob: float = 0.0,
                 rng: np.random.Generator | None = None):
        self.edge_index = edge_index
        self.n_agents = n_agents
        self.gated = gated
        self.fail_prob = fail_prob
        self.rng = rng or np.random.default_rng(0)
        self.out_degree = np.bincount(edge_index[0], minlength=n_agents) if edge_index.size else np.zeros(n_agents, int)
        self.reset()

    def reset(self) -> None:
        self.gates = np.ones(self.n_agents, dtype=np.int64)

    def active_mask(self) -> np.ndarray:
        mask = self.gates[self.edge_index[0]].astype(bool) if self.gated else np.ones(self.edge_index.shape[1], bool)
        if self.fail_prob > 0:
            mask &= self.rng.random(mask.shape[0]) >= self.fail_prob
        return mask

    def update(self, gates: np.ndarray | None) -> None:
        if self.gated and gates is not None:
            self.gates = np.asarray(gates, dtype=np.int64)

    def comm_cost(self, mask: np.ndarray) -> np.ndarray:
        """Messages sent by each agent at this step (number of active out-edges)."""
        return np.bincount(self.edge_index[0][mask], minlength=self.n_agents).astype(np.float64)


def batch_graphs(edge_index: torch.Tensor, n_nodes: int, masks: torch.Tensor,
                 edge_attr: torch.Tensor | None) -> tuple[torch.Tensor, torch.Tensor | None]:
    """Disjoint union of B copies of the same node set with per-copy edge masks.

    edge_index: (2, E) base edges; masks: (B, E) bool; edge_attr: (B, E, F) or None.
    Returns edge_index (2, E_active) over B*n_nodes nodes and matching edge_attr.
    """
    B, E = masks.shape
    offsets = (torch.arange(B, device=edge_index.device) * n_nodes).view(1, B, 1)
    ei = (edge_index.view(2, 1, E) + offsets).reshape(2, B * E)
    m = masks.reshape(B * E)
    ei = ei[:, m]
    ea = edge_attr.reshape(B * E, -1)[m] if edge_attr is not None else None
    return ei, ea

**`nashgat/models.py`** — Encoders (MLP / mean aggregation / GATv2 with edge features), decentralised actor, critics.

In [ ]:
%%writefile nashgat/models.py
"""Encoders, decentralised actors and centralised / graph critics."""
from __future__ import annotations

import torch
import torch.nn as nn
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import scatter


def mlp(i: int, h: int, o: int, layers: int = 2) -> nn.Sequential:
    mods: list[nn.Module] = []
    d = i
    for _ in range(layers - 1):
        mods += [nn.Linear(d, h), nn.ReLU()]
        d = h
    mods.append(nn.Linear(d, o))
    return nn.Sequential(*mods)


class MLPEncoder(nn.Module):
    """No communication: h_i = f(o_i)."""

    def __init__(self, obs_dim: int, hidden: int, **_):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU())
        self.out_dim = hidden

    def forward(self, x, edge_index=None, edge_attr=None, return_attention=False):
        h = self.net(x)
        return (h, None) if return_attention else h


class MeanAggEncoder(nn.Module):
    """Uniform neighbour averaging (the 'no attention' ablation): h_i = f([z_i, mean_{j in N_t(i)} z_j])."""

    def __init__(self, obs_dim: int, hidden: int, **_):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU())
        self.out = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.ReLU())
        self.out_dim = hidden

    def forward(self, x, edge_index, edge_attr=None, return_attention=False):
        z = self.inp(x)
        if edge_index.numel():
            agg = scatter(z[edge_index[0]], edge_index[1], dim=0, dim_size=z.size(0), reduce="mean")
        else:
            agg = torch.zeros_like(z)
        h = self.out(torch.cat([z, agg], -1))
        return (h, None) if return_attention else h


class GATEncoder(nn.Module):
    """One round of GATv2 message passing (one-hop communication).

    With edge features the attention logit of edge j->i is
        e_ij = a^T LeakyReLU(W_s z_j + W_t z_i + W_e x_ij),
    so the weight an agent places on a neighbour depends on both agents' states and on
    the measured traffic coupling of the road joining them (strategic-influence proxy).
    """

    def __init__(self, obs_dim: int, hidden: int, heads: int = 4, edge_dim: int | None = None, layers: int = 1, **_):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU())
        self.convs = nn.ModuleList(
            [GATv2Conv(hidden, hidden // heads, heads=heads, edge_dim=edge_dim, add_self_loops=True,
                       fill_value="mean" if edge_dim else 0.0) for _ in range(layers)])
        self.edge_dim = edge_dim
        self.out_dim = 2 * hidden

    def forward(self, x, edge_index, edge_attr=None, return_attention=False):
        z = self.inp(x)
        h = z
        att = None
        ea = edge_attr if self.edge_dim else None
        for conv in self.convs:
            if return_attention:
                h2, att = conv(h, edge_index, edge_attr=ea, return_attention_weights=True)
            else:
                h2 = conv(h, edge_index, edge_attr=ea)
            h = torch.relu(h2) + h
        out = torch.cat([z, h], -1)  # keep the agent's own features explicitly
        return (out, att) if return_attention else out


ENCODERS = {"mlp": MLPEncoder, "mean": MeanAggEncoder, "gat": GATEncoder}


class Actor(nn.Module):
    """Decentralised policy pi_theta(a_i, g_i | h_i). Parameters are shared across agents."""

    def __init__(self, obs_dim: int, n_actions: int, hidden: int, encoder: str, edge_dim: int | None,
                 heads: int = 4, comm_gate: bool = False):
        super().__init__()
        self.encoder = ENCODERS[encoder](obs_dim=obs_dim, hidden=hidden, heads=heads, edge_dim=edge_dim)
        self.pi = mlp(self.encoder.out_dim, hidden, n_actions)
        self.comm_gate = comm_gate
        self.gate = mlp(self.encoder.out_dim, hidden, 1) if comm_gate else None
        # bias the gate towards transmitting at initialisation so early learning sees messages
        if self.gate is not None:
            nn.init.constant_(self.gate[-1].bias, 2.0)

    def forward(self, x, edge_index, edge_attr=None, return_attention=False):
        h, att = self.encoder(x, edge_index, edge_attr, return_attention=True)
        logits = self.pi(h)
        gate_logit = self.gate(h).squeeze(-1) if self.gate is not None else None
        return (logits, gate_logit, att) if return_attention else (logits, gate_logit)


class LocalCritic(nn.Module):
    """IPPO critic: V_i(o_i, t) and Q_i(o_i, t, .)."""

    def __init__(self, obs_dim: int, n_agents: int, n_actions: int, hidden: int, **_):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(obs_dim + 1, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU())
        self.v = nn.Linear(hidden, 1)
        self.q = nn.Linear(hidden, n_actions)

    def forward(self, obs, tfeat, edge_index=None, edge_attr=None):
        # obs (B, N, D), tfeat (B,)
        B, N, _ = obs.shape
        h = self.trunk(torch.cat([obs, tfeat.view(B, 1, 1).expand(B, N, 1)], -1))
        return self.v(h).squeeze(-1), self.q(h)


class CentralCritic(nn.Module):
    """MAPPO critic on the concatenated global observation plus an agent one-hot."""

    def __init__(self, obs_dim: int, n_agents: int, n_actions: int, hidden: int, **_):
        super().__init__()
        self.n = n_agents
        self.trunk = nn.Sequential(nn.Linear(obs_dim * n_agents + n_agents + 1, 2 * hidden), nn.ReLU(),
                                   nn.Linear(2 * hidden, hidden), nn.ReLU())
        self.v = nn.Linear(hidden, 1)
        self.q = nn.Linear(hidden, n_actions)

    def forward(self, obs, tfeat, edge_index=None, edge_attr=None):
        B, N, D = obs.shape
        g = obs.reshape(B, 1, N * D).expand(B, N, N * D)
        eye = torch.eye(N, device=obs.device).unsqueeze(0).expand(B, N, N)
        h = self.trunk(torch.cat([g, eye, tfeat.view(B, 1, 1).expand(B, N, 1)], -1))
        return self.v(h).squeeze(-1), self.q(h)


class GraphCritic(nn.Module):
    """Graph-conditioned centralised critic: two GATv2 layers over the full static
    graph (training-time information only) plus a global mean-pooled context."""

    def __init__(self, obs_dim: int, n_agents: int, n_actions: int, hidden: int, edge_dim: int | None = None,
                 heads: int = 4, **_):
        super().__init__()
        self.enc = GATEncoder(obs_dim + 1, hidden, heads=heads, edge_dim=edge_dim, layers=2)
        d = self.enc.out_dim
        self.trunk = nn.Sequential(nn.Linear(2 * d, hidden), nn.ReLU())
        self.v = nn.Linear(hidden, 1)
        self.q = nn.Linear(hidden, n_actions)

    def forward(self, obs, tfeat, edge_index, edge_attr=None):
        B, N, D = obs.shape
        x = torch.cat([obs, tfeat.view(B, 1, 1).expand(B, N, 1)], -1).reshape(B * N, D + 1)
        h = self.enc(x, edge_index, edge_attr).view(B, N, -1)
        g = h.mean(1, keepdim=True).expand_as(h)
        z = self.trunk(torch.cat([h, g], -1))
        return self.v(z).squeeze(-1), self.q(z)


CRITICS = {"local": LocalCritic, "central": CentralCritic, "graph": GraphCritic}

In [ ]:
%%writefile nashgat/algos/__init__.py
"""Learning algorithms: PPO/MAPPO family and best-response evaluation."""

**`nashgat/algos/mappo.py`** — Unified PPO/MAPPO trainer: IPPO, MAPPO, GNN-MAPPO, Nash-MAPPO, NashGAT-MARL and ablations.

In [ ]:
%%writefile nashgat/algos/mappo.py
"""Unified PPO / MAPPO trainer covering all compared methods.

A method is a point in the design space
    actor encoder  x  critic  x  reward mode  x  graph dynamics  x  (lambda, mu):

  IPPO          mlp actor, local critic,   individual rewards
  MAPPO         mlp actor, central critic, welfare (mean) reward
  GNN-MAPPO     GAT actor (static graph, no edge features), graph critic, welfare
  Nash-MAPPO    MAPPO + exploitability-gap regulariser (lambda > 0)
  NashGAT-MARL  GAT actor with traffic-coupling edge features and learned
                communication gates (dynamic E_t), graph critic, welfare reward,
                gap regulariser (lambda > 0) and communication penalty (mu > 0)

The gap regulariser for agent i at state s is
    gap_i(s) = max_a Q_i(s, a) - sum_a pi_i(a | h_i) Q_i(s, a)  >= 0,
where Q_i is a centralised critic of agent i's *individual* reward under the current
joint policy. Its expectation under d^pi upper-bounds agent i's exploitability up to a
distribution-mismatch factor (MATHEMATICAL_MODEL.md, Prop. 1).
"""
from __future__ import annotations

import copy
import time
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Bernoulli, Categorical

from ..envs.traffic_env import TrafficMAEnv
from ..graph import DynamicGraphBuilder, batch_graphs
from ..models import CRITICS, Actor


@dataclass
class AlgoConfig:
    name: str = "nashgat"
    actor_encoder: str = "gat"  # mlp | gat | mean
    edge_features: bool = True  # traffic-coupling edge features in actor attention
    comm_gate: bool = True  # learned dynamic communication graph
    critic: str = "graph"  # local | central | graph
    reward_mode: str = "welfare"  # individual | welfare
    nash_lambda: float = 0.5
    comm_mu: float = 0.05
    hidden: int = 64
    heads: int = 4
    lr_actor: float = 5e-4
    lr_critic: float = 1e-3
    gamma: float = 0.99
    gae_lambda: float = 0.95
    clip: float = 0.2
    epochs: int = 4
    minibatches: int = 4
    ent_coef: float = 0.01
    max_grad_norm: float = 0.5
    episodes: int = 150
    comm_fail_prob: float = 0.0


PRESETS: dict[str, dict] = {
    "ippo": dict(actor_encoder="mlp", edge_features=False, comm_gate=False, critic="local",
                 reward_mode="individual", nash_lambda=0.0, comm_mu=0.0),
    "mappo": dict(actor_encoder="mlp", edge_features=False, comm_gate=False, critic="central",
                  reward_mode="welfare", nash_lambda=0.0, comm_mu=0.0),
    "gnn_mappo": dict(actor_encoder="gat", edge_features=False, comm_gate=False, critic="graph",
                      reward_mode="welfare", nash_lambda=0.0, comm_mu=0.0),
    "nash_mappo": dict(actor_encoder="mlp", edge_features=False, comm_gate=False, critic="central",
                       reward_mode="welfare", nash_lambda=0.5, comm_mu=0.0),
    "nashgat": dict(actor_encoder="gat", edge_features=True, comm_gate=True, critic="graph",
                    reward_mode="welfare", nash_lambda=0.5, comm_mu=0.05),
    # ablations of NashGAT-MARL
    "nashgat_no_gnn": dict(actor_encoder="mlp", edge_features=False, comm_gate=False, critic="graph",
                           reward_mode="welfare", nash_lambda=0.5, comm_mu=0.0),
    "nashgat_no_attention": dict(actor_encoder="mean", edge_features=False, comm_gate=True, critic="graph",
                                 reward_mode="welfare", nash_lambda=0.5, comm_mu=0.05),
    "nashgat_no_nash": dict(actor_encoder="gat", edge_features=True, comm_gate=True, critic="graph",
                            reward_mode="welfare", nash_lambda=0.0, comm_mu=0.05),
    "nashgat_no_comm_penalty": dict(actor_encoder="gat", edge_features=True, comm_gate=True, critic="graph",
                                    reward_mode="welfare", nash_lambda=0.5, comm_mu=0.0),
    "nashgat_static_graph": dict(actor_encoder="gat", edge_features=True, comm_gate=False, critic="graph",
                                 reward_mode="welfare", nash_lambda=0.5, comm_mu=0.0),
}


def make_config(preset: str, **overrides) -> AlgoConfig:
    base = AlgoConfig(name=preset)
    return replace(base, **{**PRESETS[preset], **overrides})


class MAPPOTrainer:
    def __init__(self, env: TrafficMAEnv, cfg: AlgoConfig, seed: int = 0, device: str | torch.device = "cpu"):
        self.env, self.cfg, self.seed = env, cfg, seed
        self.device = torch.device(device)
        self.N, self.A, self.D = env.n_agents, env.n_actions, env.obs_dim
        self.edge_index_np = env.edge_index
        self.edge_index = torch.as_tensor(env.edge_index, device=self.device)
        self.rng = np.random.default_rng(seed)
        self.uses_messages = cfg.actor_encoder != "mlp"
        self.graph = DynamicGraphBuilder(env.edge_index, self.N, gated=cfg.comm_gate,
                                         fail_prob=cfg.comm_fail_prob, rng=self.rng)
        edim = env.edge_dim if cfg.edge_features else None
        self.actor = Actor(self.D, self.A, cfg.hidden, cfg.actor_encoder, edim, cfg.heads, cfg.comm_gate).to(self.device)
        self.critic = CRITICS[cfg.critic](obs_dim=self.D, n_agents=self.N, n_actions=self.A, hidden=cfg.hidden,
                                          edge_dim=env.edge_dim, heads=cfg.heads).to(self.device)
        self.opt_a = torch.optim.Adam(self.actor.parameters(), lr=cfg.lr_actor)
        self.opt_c = torch.optim.Adam(self.critic.parameters(), lr=cfg.lr_critic)
        self.max_out = max(1, int(self.graph.out_degree.max()))
        self.history: list[dict] = []
        self.gen = torch.Generator(device="cpu").manual_seed(seed)

    # ------------------------------------------------------------ helpers
    def _actor_forward(self, actor: Actor, obs: torch.Tensor, masks: torch.Tensor, eattr: torch.Tensor,
                       return_attention: bool = False):
        """obs (B,N,D), masks (B,E), eattr (B,E,F) -> logits (B,N,A), gate logits (B,N) | None."""
        B = obs.shape[0]
        ei, ea = batch_graphs(self.edge_index, self.N, masks, eattr if self.cfg.edge_features else None)
        out = actor(obs.reshape(B * self.N, self.D), ei, ea, return_attention=return_attention)
        logits = out[0].view(B, self.N, self.A)
        gl = out[1].view(B, self.N) if out[1] is not None else None
        if return_attention:
            return logits, gl, out[2]
        return logits, gl

    def _critic_forward(self, obs: torch.Tensor, tfeat: torch.Tensor, eattr: torch.Tensor):
        B = obs.shape[0]
        if self.cfg.critic == "graph":
            full = torch.ones(B, self.edge_index.shape[1], dtype=torch.bool, device=self.device)
            ei, ea = batch_graphs(self.edge_index, self.N, full, eattr)
            return self.critic(obs, tfeat, ei, ea)
        return self.critic(obs, tfeat)

    # ------------------------------------------------------------ rollout
    @torch.no_grad()
    def rollout(self, greedy: bool = False, dev_actor: Actor | None = None, dev_agent: int | None = None,
                record_attention_steps: tuple[int, ...] = (), env: TrafficMAEnv | None = None) -> dict:
        env = env or self.env
        obs, _ = env.reset()
        self.graph.reset()
        T = int(np.ceil(env.cfg.horizon / env.cfg.decision_interval))
        E = self.edge_index.shape[1]
        buf = {k: [] for k in ("obs", "eattr", "mask", "act", "gate", "logp", "logits", "r", "comm", "tfeat")}
        attention = {}
        t = 0
        while True:
            mask = self.graph.active_mask() if self.uses_messages else np.zeros(E, bool)
            eattr = env.edge_features()
            o = torch.as_tensor(obs, device=self.device).unsqueeze(0)
            m = torch.as_tensor(mask, device=self.device).unsqueeze(0)
            ea = torch.as_tensor(eattr, device=self.device).unsqueeze(0)
            if t in record_attention_steps and self.cfg.actor_encoder == "gat":
                logits, gl, att = self._actor_forward(self.actor, o, m, ea, return_attention=True)
                attention[t] = (att[0].cpu().numpy(), att[1].mean(-1).cpu().numpy())
            else:
                logits, gl = self._actor_forward(self.actor, o, m, ea)
            if dev_actor is not None:
                dl, dg = self._actor_forward(dev_actor, o, m, ea)
                logits = logits.clone()
                logits[0, dev_agent] = dl[0, dev_agent]
                if gl is not None:
                    gl = gl.clone()
                    gl[0, dev_agent] = dg[0, dev_agent]
            dist = Categorical(logits=logits[0])
            a = logits[0].argmax(-1) if greedy else dist.sample()
            logp = dist.log_prob(a)
            if gl is not None:
                gd = Bernoulli(logits=gl[0])
                g = (gl[0] > 0).float() if greedy else gd.sample()
                logp = logp + gd.log_prob(g)
            else:
                g = torch.ones(self.N, device=self.device)
            comm = self.graph.comm_cost(mask) if self.uses_messages else np.zeros(self.N)
            buf["obs"].append(obs)
            buf["eattr"].append(eattr)
            buf["mask"].append(mask)
            buf["act"].append(a.cpu().numpy())
            buf["gate"].append(g.cpu().numpy())
            buf["logp"].append(logp.cpu().numpy())
            buf["logits"].append(logits[0].cpu().numpy())
            buf["comm"].append(comm)
            buf["tfeat"].append(t / T)
            self.graph.update(g.cpu().numpy().astype(np.int64))
            obs, r, _, trunc, info = env.step(a.cpu().numpy())
            buf["r"].append(r)
            t += 1
            if trunc:
                break
        out = {k: np.asarray(v) for k, v in buf.items()}
        out["last_obs"] = obs
        out["last_eattr"] = env.edge_features()
        out["last_mask"] = self.graph.active_mask() if self.uses_messages else np.zeros(E, bool)
        out["last_tfeat"] = t / T
        out["metrics"] = info["metrics"]
        out["attention"] = attention
        return out

    # --------------------------------------------------------------- update
    def _returns(self, rewards: np.ndarray, values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """GAE(lambda) with time-limit bootstrapping. rewards (T,N), values (T+1,N)."""
        cfg = self.cfg
        T = rewards.shape[0]
        adv = np.zeros_like(rewards)
        last = np.zeros(rewards.shape[1])
        for t in reversed(range(T)):
            delta = rewards[t] + cfg.gamma * values[t + 1] - values[t]
            last = delta + cfg.gamma * cfg.gae_lambda * last
            adv[t] = last
        return adv, adv + values[:-1]

    def shaped_rewards(self, buf: dict) -> np.ndarray:
        cfg = self.cfg
        r = buf["r"]
        c = buf["gate"] * self.graph.out_degree[None, :] / self.max_out if cfg.comm_gate else np.zeros_like(r)
        if cfg.reward_mode == "individual":
            return r - cfg.comm_mu * c
        team = r.mean(1, keepdims=True) - cfg.comm_mu * c.mean(1, keepdims=True)
        return np.repeat(team, self.N, axis=1)

    def update(self, buf: dict) -> dict:
        cfg, dev = self.cfg, self.device
        T = buf["r"].shape[0]
        obs_all = torch.as_tensor(np.concatenate([buf["obs"], buf["last_obs"][None]]), device=dev)
        ea_all = torch.as_tensor(np.concatenate([buf["eattr"], buf["last_eattr"][None]]), device=dev)
        tf_all = torch.as_tensor(np.append(buf["tfeat"], buf["last_tfeat"]), dtype=torch.float32, device=dev)
        mask_all = torch.as_tensor(np.concatenate([buf["mask"], buf["last_mask"][None]]), device=dev)
        with torch.no_grad():
            v_all, q_all = self._critic_forward(obs_all, tf_all, ea_all)
            last_logits, _ = self._actor_forward(self.actor, obs_all[-1:], mask_all[-1:], ea_all[-1:])
            pi_all = torch.softmax(torch.cat([torch.as_tensor(buf["logits"], device=dev), last_logits]), -1)
            v_ind = (pi_all * q_all).sum(-1)
        R = self.shaped_rewards(buf)
        adv, ret = self._returns(R, v_all.cpu().numpy())
        _, q_ret = self._returns(buf["r"], v_ind.cpu().numpy())
        adv_std = float(adv.std()) + 1e-8
        adv_n = (adv - adv.mean()) / adv_std

        obs = obs_all[:-1]
        ea = ea_all[:-1]
        tf = tf_all[:-1]
        masks = mask_all[:-1]
        act = torch.as_tensor(buf["act"], device=dev)
        gate = torch.as_tensor(buf["gate"], dtype=torch.float32, device=dev)
        old_logp = torch.as_tensor(buf["logp"], device=dev)
        adv_t = torch.as_tensor(adv_n, dtype=torch.float32, device=dev)
        ret_t = torch.as_tensor(ret, dtype=torch.float32, device=dev)
        qret_t = torch.as_tensor(q_ret, dtype=torch.float32, device=dev)
        stats = {"pg": 0.0, "v": 0.0, "q": 0.0, "gap": 0.0, "ent": 0.0, "kl": 0.0}
        n_mb = 0
        for _ in range(cfg.epochs):
            perm = torch.randperm(T, generator=self.gen)
            for idx in perm.chunk(cfg.minibatches):
                idx = idx.to(dev)
                logits, gl = self._actor_forward(self.actor, obs[idx], masks[idx], ea[idx])
                dist = Categorical(logits=logits)
                logp = dist.log_prob(act[idx])
                ent = dist.entropy()
                if gl is not None:
                    gd = Bernoulli(logits=gl)
                    logp = logp + gd.log_prob(gate[idx])
                    ent = ent + gd.entropy()
                ratio = torch.exp(logp - old_logp[idx])
                a_mb = adv_t[idx]
                pg = -torch.min(ratio * a_mb, torch.clamp(ratio, 1 - cfg.clip, 1 + cfg.clip) * a_mb).mean()
                v, q = self._critic_forward(obs[idx], tf[idx], ea[idx])
                qd = q.detach()
                pi = torch.softmax(logits, -1)
                gap = qd.max(-1).values - (pi * qd).sum(-1)  # (B,N), >= 0
                loss_a = pg - cfg.ent_coef * ent.mean()
                if cfg.nash_lambda > 0:
                    loss_a = loss_a + cfg.nash_lambda * gap.mean() / adv_std
                self.opt_a.zero_grad()
                loss_a.backward()
                nn.utils.clip_grad_norm_(self.actor.parameters(), cfg.max_grad_norm)
                self.opt_a.step()
                q_taken = q.gather(-1, act[idx].unsqueeze(-1)).squeeze(-1)
                loss_v = ((v - ret_t[idx]) ** 2).mean()
                loss_q = ((q_taken - qret_t[idx]) ** 2).mean()
                self.opt_c.zero_grad()
                (loss_v + loss_q).backward()
                nn.utils.clip_grad_norm_(self.critic.parameters(), cfg.max_grad_norm)
                self.opt_c.step()
                with torch.no_grad():
                    stats["kl"] += float((old_logp[idx] - logp).mean())
                stats["pg"] += float(pg.detach())
                stats["v"] += float(loss_v.detach())
                stats["q"] += float(loss_q.detach())
                stats["gap"] += float(gap.mean().detach())
                stats["ent"] += float(ent.mean().detach())
                n_mb += 1
        return {k: v / n_mb for k, v in stats.items()}

    # ---------------------------------------------------------------- loops
    def train(self, episodes: int | None = None, log_every: int = 10, callback=None) -> list[dict]:
        episodes = episodes or self.cfg.episodes
        for ep in range(episodes):
            t0 = time.time()
            buf = self.rollout()
            st = self.update(buf)
            rec = {"episode": ep, "seed": self.seed, "method": self.cfg.name,
                   "ep_return_ind_mean": float(buf["r"].sum(0).mean()),
                   "ep_return_team": float(buf["r"].mean(1).sum()),
                   "comm_per_agent_step": float(buf["comm"].mean()),
                   "gate_rate": float(buf["gate"].mean()),
                   **{f"loss_{k}": v for k, v in st.items()},
                   **{k: v for k, v in buf["metrics"].items() if k != "time"},
                   "wall_s": time.time() - t0}
            self.history.append(rec)
            if callback:
                callback(rec)
        return self.history

    def evaluate(self, n_episodes: int = 3, greedy: bool = False, env: TrafficMAEnv | None = None) -> dict:
        """Average traffic metrics, discounted individual returns J_i and comm cost."""
        mets, J, comm, queue = [], [], [], []
        env = env or self.env
        for _ in range(n_episodes):
            b = self.rollout(greedy=greedy, env=env)
            disc = self.cfg.gamma ** np.arange(b["r"].shape[0])
            J.append((disc[:, None] * b["r"]).sum(0))
            mets.append(b["metrics"])
            comm.append(b["comm"].mean())
            queue.append(-b["r"].mean() * env.cfg.reward_scale)
        keys = [k for k in mets[0] if k != "time"]
        out = {k: float(np.mean([m[k] for m in mets])) for k in keys}
        out["J_per_agent"] = np.mean(J, 0).tolist()
        out["J_sum"] = float(np.mean(J, 0).sum())
        out["comm_per_agent_step"] = float(np.mean(comm))
        out["avg_queue_per_intersection"] = float(np.mean(queue))
        return out

    def save(self, path: str | Path) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({"cfg": asdict(self.cfg), "actor": self.actor.state_dict(), "critic": self.critic.state_dict(),
                    "seed": self.seed}, path)

    def load(self, path: str | Path) -> None:
        ck = torch.load(path, map_location=self.device, weights_only=False)
        self.actor.load_state_dict(ck["actor"])
        self.critic.load_state_dict(ck["critic"])

**`nashgat/algos/best_response.py`** — Approximate best-response learning and sampled NashConv (exploitability lower-bound estimate).

In [ ]:
%%writefile nashgat/algos/best_response.py
"""Approximate best-response learning and empirical exploitability for the traffic game.

For a trained joint policy pi and an agent k, a deviation policy pi_k' (same
architecture and information as pi_k, initialised from pi_k) is trained with PPO
on agent k's individual reward while every other agent keeps pi_-k frozen. Then

    delta_k = J_k(pi_k', pi_-k) - J_k(pi)

is estimated from independent evaluation episodes. Because pi_k' is only an
approximate best response, delta_k estimates a LOWER bound on agent k's true
exploitability within the policy class; sum_k delta_k over the sampled agents is
reported as "approximate NashConv (sampled agents)".
"""
from __future__ import annotations

import copy

import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Bernoulli, Categorical

from .mappo import MAPPOTrainer


class _ValueNet(nn.Module):
    def __init__(self, in_dim: int, hidden: int):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, 2 * hidden), nn.ReLU(), nn.Linear(2 * hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, 1))

    def forward(self, x):
        return self.net(x).squeeze(-1)


def _discounted(r: np.ndarray, gamma: float) -> float:
    return float((gamma ** np.arange(len(r)) * r).sum())


def estimate_J(tr: MAPPOTrainer, agent: int, n_episodes: int, dev_actor=None) -> list[float]:
    out = []
    for _ in range(n_episodes):
        b = tr.rollout(dev_actor=dev_actor, dev_agent=agent if dev_actor is not None else None)
        out.append(_discounted(b["r"][:, agent], tr.cfg.gamma))
    return out


def approximate_best_response(tr: MAPPOTrainer, agent: int, episodes: int = 20, eval_episodes: int = 5,
                              lr: float = 5e-4, seed: int = 0) -> dict:
    cfg, dev = tr.cfg, tr.device
    torch.manual_seed(seed)
    base_J = estimate_J(tr, agent, eval_episodes)
    dev_actor = copy.deepcopy(tr.actor)
    vnet = _ValueNet(tr.N * tr.D + 1, cfg.hidden).to(dev)
    opt_a = torch.optim.Adam(dev_actor.parameters(), lr=lr)
    opt_v = torch.optim.Adam(vnet.parameters(), lr=1e-3)
    curve = []
    for ep in range(episodes):
        b = tr.rollout(dev_actor=dev_actor, dev_agent=agent)
        T = b["r"].shape[0]
        obs_all = torch.as_tensor(np.concatenate([b["obs"], b["last_obs"][None]]), device=dev)
        tf_all = torch.as_tensor(np.append(b["tfeat"], b["last_tfeat"]), dtype=torch.float32, device=dev)
        g_in = torch.cat([obs_all.reshape(T + 1, -1), tf_all.unsqueeze(-1)], -1)
        with torch.no_grad():
            v = vnet(g_in).cpu().numpy()
        r = b["r"][:, agent]
        adv = np.zeros(T)
        last = 0.0
        for t in reversed(range(T)):
            delta = r[t] + cfg.gamma * v[t + 1] - v[t]
            last = delta + cfg.gamma * cfg.gae_lambda * last
            adv[t] = last
        ret = torch.as_tensor(adv + v[:-1], dtype=torch.float32, device=dev)
        adv_t = torch.as_tensor((adv - adv.mean()) / (adv.std() + 1e-8), dtype=torch.float32, device=dev)
        obs = obs_all[:-1]
        masks = torch.as_tensor(b["mask"], device=dev)
        ea = torch.as_tensor(b["eattr"], device=dev)
        act = torch.as_tensor(b["act"][:, agent], device=dev)
        gate = torch.as_tensor(b["gate"][:, agent], dtype=torch.float32, device=dev)
        old_logp = torch.as_tensor(b["logp"][:, agent], device=dev)
        for _ in range(cfg.epochs):
            for idx in torch.randperm(T).chunk(cfg.minibatches):
                logits, gl = tr._actor_forward(dev_actor, obs[idx], masks[idx], ea[idx])
                dist = Categorical(logits=logits[:, agent])
                logp = dist.log_prob(act[idx])
                ent = dist.entropy()
                if gl is not None:
                    gd = Bernoulli(logits=gl[:, agent])
                    logp = logp + gd.log_prob(gate[idx])
                ratio = torch.exp(logp - old_logp[idx])
                pg = -torch.min(ratio * adv_t[idx], torch.clamp(ratio, 1 - cfg.clip, 1 + cfg.clip) * adv_t[idx]).mean()
                loss = pg - cfg.ent_coef * ent.mean()
                opt_a.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(dev_actor.parameters(), cfg.max_grad_norm)
                opt_a.step()
                lv = ((vnet(g_in[:-1][idx]) - ret[idx]) ** 2).mean()
                opt_v.zero_grad()
                lv.backward()
                opt_v.step()
        curve.append(_discounted(r, cfg.gamma))
    br_J = estimate_J(tr, agent, eval_episodes, dev_actor=dev_actor)
    gain = np.array(br_J).mean() - np.array(base_J).mean()
    se = np.sqrt(np.var(br_J, ddof=1) / len(br_J) + np.var(base_J, ddof=1) / len(base_J)) if eval_episodes > 1 else np.nan
    return {"agent": agent, "J_base": float(np.mean(base_J)), "J_br": float(np.mean(br_J)), "gain": float(gain),
            "gain_se": float(se), "train_curve": curve}


def sampled_nash_conv(tr: MAPPOTrainer, agents: list[int], **kw) -> dict:
    res = [approximate_best_response(tr, k, seed=tr.seed * 100 + k, **kw) for k in agents]
    gains = np.array([r["gain"] for r in res])
    return {"agents": agents, "per_agent": res, "nash_conv_sampled": float(np.maximum(gains, 0).sum()),
            "nash_conv_sampled_raw": float(gains.sum())}

**`nashgat/rule_based.py`** — FixedTime and MaxPressure reference controllers.

In [ ]:
%%writefile nashgat/rule_based.py
"""Non-learning reference controllers (FixedTime, MaxPressure) on the same simulator."""
from __future__ import annotations

import numpy as np

from .envs.traffic_env import TrafficMAEnv


def _phase_pressures(env: TrafficMAEnv, a: int) -> np.ndarray:
    sim = env.sim
    q = sim.queue_lengths()
    out_load = np.zeros(sim.n_mov)
    for m, (_, r_out) in enumerate(sim.mov_key):
        out_load[m] = sim.count[r_out] / sim.cap[r_out] * 10.0  # downstream occupancy proxy
    pressures = []
    for p in range(1, env.n_actions + 1):
        movs = sim.agent_phases[a][p]
        pressures.append(sum(q[m] - out_load[m] for m in movs))
    return np.array(pressures)


def run_rule(env: TrafficMAEnv, policy: str, cycle_steps: int = 3, gamma: float = 0.99) -> dict:
    """policy: 'fixed' (each phase for cycle_steps decisions) or 'maxpressure'."""
    env.reset()
    k, rs = 0, []
    while True:
        if policy == "fixed":
            act = np.full(env.n_agents, (k // cycle_steps) % env.n_actions)
        elif policy == "maxpressure":
            act = np.array([int(np.argmax(_phase_pressures(env, a))) for a in range(env.n_agents)])
        else:
            raise ValueError(policy)
        _, r, _, trunc, info = env.step(act)
        rs.append(r)
        k += 1
        if trunc:
            break
    R = np.array(rs)
    out = {kk: v for kk, v in info["metrics"].items() if kk != "time"}
    out["J_sum"] = float((gamma ** np.arange(len(R))[:, None] * R).sum())
    out["avg_queue_per_intersection"] = float(-R.mean() * env.cfg.reward_scale)
    # MaxPressure needs downstream state from every neighbour at every decision
    out["comm_per_agent_step"] = env.edge_index.shape[1] / env.n_agents if policy == "maxpressure" else 0.0
    return out

**`nashgat/stats.py`** — Mean, std, t-intervals, Welch / Mann-Whitney tests, Holm correction.

In [ ]:
%%writefile nashgat/stats.py
"""Statistical summaries for multi-seed comparisons."""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats


def mean_ci(x: np.ndarray, conf: float = 0.95) -> tuple[float, float, float, float]:
    """Mean, sample std, and Student-t confidence interval half-width bounds."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    m = float(x.mean())
    s = float(x.std(ddof=1)) if n > 1 else float("nan")
    if n < 2:
        return m, s, float("nan"), float("nan")
    h = stats.t.ppf(0.5 + conf / 2, n - 1) * s / np.sqrt(n)
    return m, s, m - h, m + h


def welch(a: np.ndarray, b: np.ndarray) -> tuple[float, float]:
    r = stats.ttest_ind(a, b, equal_var=False)
    return float(r.statistic), float(r.pvalue)


def mann_whitney(a: np.ndarray, b: np.ndarray) -> float:
    return float(stats.mannwhitneyu(a, b, alternative="two-sided").pvalue)


def holm(pvals: list[float]) -> list[float]:
    """Holm-Bonferroni adjusted p-values (same order as input)."""
    p = np.asarray(pvals, dtype=float)
    order = np.argsort(p)
    m = len(p)
    adj = np.empty(m)
    running = 0.0
    for k, i in enumerate(order):
        running = max(running, min(1.0, (m - k) * p[i]))
        adj[i] = running
    return adj.tolist()


def summarise(df: pd.DataFrame, group: str, metrics: list[str]) -> pd.DataFrame:
    rows = []
    for g, d in df.groupby(group, sort=False):
        row = {group: g, "n": len(d)}
        for m in metrics:
            v = d[m].dropna().to_numpy()
            if len(v) == 0:
                row.update({f"{m}_mean": float("nan"), f"{m}_std": float("nan"), f"{m}_ci_lo": float("nan"),
                            f"{m}_ci_hi": float("nan"), f"{m}_n": 0})
                continue
            mu, sd, lo, hi = mean_ci(v)
            row[f"{m}_n"] = len(v)
            row.update({f"{m}_mean": mu, f"{m}_std": sd, f"{m}_ci_lo": lo, f"{m}_ci_hi": hi})
        rows.append(row)
    return pd.DataFrame(rows)


def compare_to(df: pd.DataFrame, group: str, ref: str, metrics: list[str], lower_is_better: dict[str, bool]) -> pd.DataFrame:
    """Relative change of `ref` against every other group with Welch and Mann-Whitney
    tests; Holm correction is applied per metric across the comparisons."""
    rows = []
    a_all = df[df[group] == ref]
    for m in metrics:
        tmp = []
        for g, d in df.groupby(group, sort=False):
            if g == ref:
                continue
            a, b = a_all[m].dropna().to_numpy(), d[m].dropna().to_numpy()
            if len(a) < 2 or len(b) < 2:
                continue
            t, p = welch(a, b)
            rel = (a.mean() - b.mean()) / abs(b.mean()) * 100 if b.mean() != 0 else float("nan")
            better = (a.mean() < b.mean()) if lower_is_better.get(m, False) else (a.mean() > b.mean())
            tmp.append({"metric": m, "vs": g, "ref_mean": a.mean(), "other_mean": b.mean(), "rel_change_%": rel,
                        "ref_better": bool(better), "welch_t": t, "p_welch": p, "p_mwu": mann_whitney(a, b)})
        adj = holm([r["p_welch"] for r in tmp])
        for r, q in zip(tmp, adj):
            r["p_welch_holm"] = q
        rows += tmp
    return pd.DataFrame(rows)

**`nashgat/experiment.py`** — Single-run driver: train, evaluate (nominal + robustness + transfer), best response.

In [ ]:
%%writefile nashgat/experiment.py
"""Single-run experiment driver shared by scripts and the Colab notebook."""
from __future__ import annotations

import time
from dataclasses import asdict, replace
from pathlib import Path

import numpy as np
import pandas as pd

from .algos.best_response import sampled_nash_conv
from .algos.mappo import MAPPOTrainer, make_config
from .envs.traffic_env import TrafficMAEnv, default_env_config
from .utils import save_json, set_seed


def evaluate_suite(tr: MAPPOTrainer, data_root: str | Path, eval_episodes: int = 3, robustness: bool = True) -> dict:
    res = {}
    base = tr.env.cfg
    res["nominal_stochastic"] = tr.evaluate(eval_episodes)
    res["nominal_greedy"] = tr.evaluate(1, greedy=True)
    if not robustness:
        return res
    res["obs_noise_0.1"] = tr.evaluate(eval_episodes, env=TrafficMAEnv(replace(base, obs_noise_std=0.1, seed=7)))
    old = tr.graph.fail_prob
    tr.graph.fail_prob = 0.3
    res["comm_fail_0.3"] = tr.evaluate(eval_episodes)
    tr.graph.fail_prob = old
    for fv in ("2000", "2500"):
        cfg = replace(default_env_config(data_root, "jinan", fv, horizon=base.horizon), backend=base.backend)
        res[f"flow_{fv}"] = tr.evaluate(eval_episodes, env=TrafficMAEnv(cfg))
    res["demand_x1.3"] = tr.evaluate(eval_episodes,
                                     env=TrafficMAEnv(replace(base, demand_scale=1.3, departure_jitter=30, seed=11)))
    res["agent_fail_2"] = tr.evaluate(eval_episodes, env=TrafficMAEnv(replace(base, failed_agents=[4, 7])))
    hz = TrafficMAEnv(replace(default_env_config(data_root, "hangzhou", "real", horizon=base.horizon), backend=base.backend))
    tr_hz = MAPPOTrainer(hz, replace(tr.cfg, critic="local"), seed=tr.seed)
    tr_hz.actor.load_state_dict(tr.actor.state_dict())
    res["transfer_hangzhou"] = tr_hz.evaluate(eval_episodes)
    return res


def run_single(method: str, seed: int, data_root: str | Path, out_root: str | Path, episodes: int,
               horizon: int = 3600, backend: str = "queuesim", eval_episodes: int = 3, robustness: bool = True,
               br_agents: tuple[int, ...] = (0, 1, 5), br_episodes: int = 20, br_eval_episodes: int = 5,
               device: str = "cpu", tag: str = "", log_every: int = 0, **overrides) -> dict:
    set_seed(seed)
    cfg = make_config(method, episodes=episodes, **overrides)
    label = method + (f"_{tag}" if tag else "")
    cfg = replace(cfg, name=label)
    out = Path(out_root) / label / f"seed{seed}"
    out.mkdir(parents=True, exist_ok=True)
    env = TrafficMAEnv(replace(default_env_config(data_root, "jinan", "real", horizon=horizon, seed=seed), backend=backend))
    tr = MAPPOTrainer(env, cfg, seed=seed, device=device)
    save_json({"algo": asdict(cfg), "env": asdict(env.cfg)}, out / "config.json")
    t0 = time.time()

    def cb(rec):
        if log_every and rec["episode"] % log_every == 0:
            print(f"  [{label} s{seed}] ep {rec['episode']:4d} team_return {rec['ep_return_team']:8.1f} "
                  f"ATT {rec['avg_travel_time']:6.1f} gap {rec['loss_gap']:.3f} msgs {rec['comm_per_agent_step']:.2f}")

    tr.train(episodes, callback=cb)
    train_time = time.time() - t0
    pd.DataFrame(tr.history).to_csv(out / "history.csv", index=False)
    tr.save(out / "model.pt")
    ev = evaluate_suite(tr, data_root, eval_episodes, robustness)
    ev["train_time_s"] = train_time
    ev["n_actor_params"] = sum(p.numel() for p in tr.actor.parameters())
    save_json(ev, out / "eval.json")
    if cfg.actor_encoder == "gat":
        T = int(np.ceil(horizon / env.cfg.decision_interval))
        steps = tuple(sorted({T // 6, T // 2, (5 * T) // 6}))
        b = tr.rollout(record_attention_steps=steps)
        np.savez(out / "attention.npz", **{f"t{t}_edges": v[0] for t, v in b["attention"].items()},
                 **{f"t{t}_alpha": v[1] for t, v in b["attention"].items()},
                 **{f"t{t}_eattr": b["eattr"][t] for t in b["attention"]})
    if br_agents:
        nc = sampled_nash_conv(tr, list(br_agents), episodes=br_episodes, eval_episodes=br_eval_episodes)
        save_json(nc, out / "nashconv.json")
    return {"label": label, "seed": seed, "trainer": tr, "eval": ev}

**`nashgat/report.py`** — Aggregation into CSV tables and publication figures.

In [ ]:
%%writefile nashgat/report.py
"""Aggregate experiment outputs into CSV tables, statistics and figures.

Reads <results>/traffic/**, <results>/rule_based.csv, <results>/graphical_games/*.csv
and writes <results>/tables/*.csv and <figures>/*.png|pdf. Only numbers present in
those files are used. Call `configure()` to change the directories.
"""
from __future__ import annotations

import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

from .stats import compare_to, mean_ci, summarise  # noqa: E402

ROOT = Path(__file__).resolve().parents[1]
RES = ROOT / "results"
TAB = RES / "tables"
FIG = ROOT / "results/figures"
DATA_ROOT = ROOT


def configure(results: str | Path, figures: str | Path, data_root: str | Path | None = None) -> None:
    global RES, TAB, FIG, DATA_ROOT
    RES, FIG = Path(results), Path(figures)
    TAB = RES / "tables"
    if data_root is not None:
        DATA_ROOT = Path(data_root)
    TAB.mkdir(parents=True, exist_ok=True)
    FIG.mkdir(parents=True, exist_ok=True)

# fixed categorical order (validated reference palette, light mode)
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
MAIN = ["ippo", "mappo", "gnn_mappo", "nash_mappo", "nashgat"]
NAMES = {"ippo": "IPPO", "mappo": "MAPPO", "gnn_mappo": "GNN-MAPPO", "nash_mappo": "Nash-MAPPO",
         "nashgat": "NashGAT-MARL", "nashgat_no_gnn": "w/o GNN", "nashgat_no_attention": "w/o attention (mean agg.)",
         "nashgat_no_nash": "w/o Nash reg. (λ=0)", "nashgat_no_comm_penalty": "w/o comm. penalty (μ=0)",
         "nashgat_static_graph": "static graph", "nashgat_lam0.1": "λ=0.1", "nashgat_lam2.0": "λ=2.0",
         "nashgat_mu0.2": "μ=0.2", "nash_mappo_lam2.0": "Nash-MAPPO λ=2.0"}
COLOR = {m: PALETTE[k] for k, m in enumerate(MAIN)}
METRICS = ["avg_travel_time", "avg_waiting_time", "avg_queue_per_intersection", "throughput", "J_sum",
           "comm_per_agent_step"]
LOWER = {"avg_travel_time": True, "avg_waiting_time": True, "avg_queue_per_intersection": True,
         "nash_conv_sampled": True, "nash_conv_sampled_raw": True, "comm_per_agent_step": True}

plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e1", "grid.linewidth": 0.6, "axes.edgecolor": "#8a8984",
                     "axes.labelcolor": "#2b2b29", "xtick.color": "#52514e", "ytick.color": "#52514e",
                     "legend.frameon": False, "savefig.dpi": 300, "figure.dpi": 110})


def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG / f"{name}.png", bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)


def load_runs() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rows, hist, br = [], [], []
    for ev in sorted((RES / "traffic").glob("*/seed*/eval.json")):
        label, seed = ev.parent.parent.name, int(ev.parent.name[4:])
        d = json.loads(ev.read_text())
        for scen, m in d.items():
            if isinstance(m, dict):
                rows.append({"method": label, "seed": seed, "scenario": scen,
                             **{k: v for k, v in m.items() if not isinstance(v, list)}})
        nc = ev.parent / "nashconv.json"
        if nc.exists():
            n = json.loads(nc.read_text())
            for p in n["per_agent"]:
                br.append({"method": label, "seed": seed, "agent": p["agent"], "gain": p["gain"], "gain_se": p["gain_se"],
                           "J_base": p["J_base"], "J_br": p["J_br"]})
            rows.append({"method": label, "seed": seed, "scenario": "nashconv",
                         "nash_conv_sampled": n["nash_conv_sampled"], "nash_conv_sampled_raw": n["nash_conv_sampled_raw"]})
        rows.append({"method": label, "seed": seed, "scenario": "cost", "train_time_s": d.get("train_time_s"),
                     "n_actor_params": d.get("n_actor_params")})
        h = pd.read_csv(ev.parent / "history.csv")
        h["method"], h["seed"] = label, seed
        hist.append(h)
    return pd.DataFrame(rows), (pd.concat(hist) if hist else pd.DataFrame()), pd.DataFrame(br)


def wide(df: pd.DataFrame) -> pd.DataFrame:
    """One row per (method, seed) with nominal metrics, NashConv, cost and final-gap columns."""
    nom = df[df.scenario == "nominal_stochastic"].set_index(["method", "seed"])[METRICS]
    gr = df[df.scenario == "nominal_greedy"].set_index(["method", "seed"])[["avg_travel_time", "throughput"]]
    gr.columns = ["greedy_" + c for c in gr.columns]
    nc = df[df.scenario == "nashconv"].set_index(["method", "seed"])[["nash_conv_sampled", "nash_conv_sampled_raw"]]
    cost = df[df.scenario == "cost"].set_index(["method", "seed"])[["train_time_s", "n_actor_params"]]
    return nom.join(gr, how="left").join(nc, how="left").join(cost, how="left").reset_index()


def main() -> None:
    TAB.mkdir(parents=True, exist_ok=True)
    FIG.mkdir(parents=True, exist_ok=True)
    df, hist, br = load_runs()
    if df.empty:
        print("no traffic results yet")
    else:
        df.to_csv(TAB / "all_runs_long.csv", index=False)
        hist.to_csv(TAB / "training_history.csv", index=False)
        br.to_csv(TAB / "best_response_per_agent.csv", index=False)
        W = wide(df)
        last = hist[hist.episode >= hist.episode.max() - 19].groupby(["method", "seed"]).agg(
            final_train_gap=("loss_gap", "mean"), final_gate_rate=("gate_rate", "mean")).reset_index()
        W = W.merge(last, on=["method", "seed"], how="left")
        W.to_csv(TAB / "per_seed_nominal.csv", index=False)
        cols = METRICS + ["greedy_avg_travel_time", "nash_conv_sampled", "nash_conv_sampled_raw", "train_time_s",
                          "final_train_gap"]
        S = summarise(W, "method", [c for c in cols if c in W])
        S.insert(1, "name", S.method.map(NAMES))
        S.to_csv(TAB / "summary_all_methods.csv", index=False)
        main_df = W[W.method.isin(MAIN)]
        if main_df.method.nunique() > 1 and "nashgat" in set(main_df.method):
            cmp = compare_to(main_df, "method", "nashgat",
                             ["avg_travel_time", "avg_waiting_time", "J_sum", "nash_conv_sampled", "comm_per_agent_step"],
                             LOWER)
            cmp.to_csv(TAB / "stat_tests_nashgat_vs_baselines.csv", index=False)
        # robustness
        rob = df[~df.scenario.isin(["nashconv", "cost"])]
        R = rob.groupby(["method", "scenario"]).agg(att_mean=("avg_travel_time", "mean"), att_std=("avg_travel_time", "std"),
                                                     n=("seed", "nunique"), thr_mean=("throughput", "mean"),
                                                     queue_mean=("avg_queue_per_intersection", "mean")).reset_index()
        R.to_csv(TAB / "robustness.csv", index=False)
        figures(W, hist, R, br)
    rb = RES / "rule_based.csv"
    if rb.exists():
        pd.read_csv(rb).to_csv(TAB / "rule_based.csv", index=False)
    graphical_figures()
    print("aggregation complete")


def _band(ax, h, metric, m, smooth=10):
    piv = h[h.method == m].pivot_table(index="episode", columns="seed", values=metric)
    if piv.empty:
        return
    piv = piv.rolling(smooth, min_periods=1).mean()
    mu = piv.mean(1)
    n = piv.notna().sum(1)
    se = piv.std(1) / np.sqrt(n)
    ax.plot(mu.index, mu, color=COLOR[m], lw=2, label=f"{NAMES[m]} (n={int(n.max())})")
    ax.fill_between(mu.index, mu - 1.96 * se, mu + 1.96 * se, color=COLOR[m], alpha=0.15, lw=0)


def figures(W, hist, R, br):
    ms = [m for m in MAIN if m in set(W.method)]
    # training curves
    fig, axs = plt.subplots(1, 3, figsize=(10, 3))
    for m in ms:
        _band(axs[0], hist, "ep_return_team", m)
        _band(axs[1], hist, "avg_travel_time", m)
        _band(axs[2], hist, "loss_gap", m)
    axs[0].set(title="Team return (training, 10-ep. mean)", xlabel="episode", ylabel="Σ_t mean_i r_i")
    axs[1].set(title="Average travel time (training)", xlabel="episode", ylabel="s")
    axs[2].set(title="Critic exploitability gap Ĝ", xlabel="episode", ylabel="mean gap (return units)")
    axs[0].legend(fontsize=7, loc="lower right")
    save(fig, "training_curves")
    # congestion bars with per-seed dots
    fig, axs = plt.subplots(1, 4, figsize=(11, 2.8))
    for ax, (c, t) in zip(axs, [("avg_travel_time", "Avg. travel time (s)"), ("avg_waiting_time", "Avg. waiting time (s)"),
                                ("avg_queue_per_intersection", "Queue / intersection (veh)"), ("throughput", "Throughput (veh)")]):
        for k, m in enumerate(ms):
            v = W[W.method == m][c].dropna().to_numpy()
            if len(v) == 0:
                continue
            mu, sd, lo, hi = mean_ci(v)
            ax.bar(k, mu, color=COLOR[m], width=0.7, yerr=None if np.isnan(lo) else [[mu - lo], [hi - mu]],
                   capsize=3, error_kw={"lw": 1, "ecolor": "#2b2b29"})
            ax.scatter(np.full(len(v), k) + np.linspace(-0.15, 0.15, len(v)), v, s=9, color="#2b2b29", zorder=3)
        ax.set_xticks(range(len(ms)), [NAMES[m] for m in ms], rotation=35, ha="right", fontsize=7)
        ax.set_title(t, fontsize=9)
    save(fig, "congestion_comparison")
    # exploitability vs welfare and comm vs welfare
    fig, axs = plt.subplots(1, 2, figsize=(8, 3))
    for m in ms:  # ablations are shown in their own figure
        d = W[W.method == m]
        col = COLOR.get(m, "#9a9994")
        mk = "o" if m in MAIN else "s"
        if d.nash_conv_sampled.notna().any():
            axs[0].errorbar(d.J_sum.mean(), d.nash_conv_sampled.mean(), xerr=d.J_sum.std(), yerr=d.nash_conv_sampled.std(),
                            fmt=mk, color=col, ms=6, capsize=2, label=NAMES.get(m, m))
        axs[1].errorbar(d.comm_per_agent_step.mean(), d.avg_travel_time.mean(), yerr=d.avg_travel_time.std(), fmt=mk,
                        color=col, ms=6, capsize=2, label=NAMES.get(m, m))
    axs[0].set(xlabel="Σ_i J_i (discounted, higher = better)", ylabel="Sampled BR NashConv (3 agents)",
               title="Welfare vs. measured exploitability")
    axs[1].set(xlabel="messages per agent per decision", ylabel="avg. travel time (s)", title="Communication vs. congestion")
    axs[1].legend(fontsize=6, loc="upper left", bbox_to_anchor=(1.0, 1.0))
    save(fig, "tradeoffs")
    # robustness heatmap-like table figure
    scen = ["nominal_stochastic", "nominal_greedy", "obs_noise_0.1", "comm_fail_0.3", "flow_2000", "flow_2500",
            "demand_x1.3", "agent_fail_2", "transfer_hangzhou"]
    P = R[R.method.isin(ms)].pivot(index="method", columns="scenario", values="att_mean").reindex(index=ms)
    P = P[[s for s in scen if s in P.columns]]
    fig, ax = plt.subplots(figsize=(9, 2.6))
    rel = P / P.min(0)
    im = ax.imshow(rel.to_numpy(), cmap="Blues", aspect="auto", vmin=1.0, vmax=max(1.3, float(np.nanmax(rel.to_numpy()))))
    for i in range(P.shape[0]):
        for j in range(P.shape[1]):
            v = P.iloc[i, j]
            if np.isfinite(v):
                ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7,
                        color="white" if rel.iloc[i, j] > 1.2 else "#0b0b0b")
    ax.set_xticks(range(P.shape[1]), P.columns, rotation=30, ha="right", fontsize=7)
    ax.set_yticks(range(P.shape[0]), [NAMES[m] for m in P.index], fontsize=7)
    ax.grid(False)
    ax.set_title("Average travel time (s) per scenario; shade = ratio to best method in column", fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.02)
    save(fig, "robustness")
    # ablations and sweeps
    abl = [m for m in ["nashgat", "nashgat_no_gnn", "nashgat_no_attention", "nashgat_no_nash", "nashgat_no_comm_penalty",
                       "nashgat_static_graph", "nashgat_lam0.1", "nashgat_lam2.0", "nashgat_mu0.2", "nash_mappo",
                       "nash_mappo_lam2.0"] if m in set(W.method)]
    if len(abl) > 1:
        # ablations ran on seeds 0-2: compare every variant on those seeds only
        Wa = W[W.seed < 3]
        fig, axs = plt.subplots(1, 3, figsize=(11, 3.2))
        for ax, (c, t, cap) in zip(axs, [("avg_travel_time", "Avg. travel time (s), seeds 0-2", 340),
                                         ("nash_conv_sampled", "Sampled BR NashConv", 40),
                                         ("comm_per_agent_step", "Messages / agent / step", None)]):
            for k, m in enumerate(abl):
                v = Wa[(Wa.method == m)][c].dropna().to_numpy()
                if len(v) == 0:
                    continue
                col = PALETTE[4] if m == "nashgat" else "#6f6e69"
                shown = np.minimum(v, cap) if cap else v
                ax.scatter(shown, np.full(len(v), k), s=16, color=col, zorder=3)
                ax.plot([v.mean()] * 2 if not cap or v.mean() <= cap else [cap] * 2, [k - 0.3, k + 0.3], color=col, lw=2)
                if cap is not None and (v > cap).any():
                    ax.annotate(", ".join(f"{x:.0f}" for x in v[v > cap]) + " →", (cap, k), xytext=(-4, 5),
                                textcoords="offset points", ha="right", fontsize=6, color="#52514e")
            if cap is not None:
                lo = np.nanmin(Wa[Wa.method.isin(abl)][c]) if c == "avg_travel_time" else 0
                ax.set_xlim(lo - 5 if c == "avg_travel_time" else -1, cap)
            ax.set_yticks(range(len(abl)), [NAMES.get(m, m) for m in abl], fontsize=7)
            ax.invert_yaxis()
            ax.set_title(t, fontsize=9)
        save(fig, "ablations")
    # BR per-agent gains
    if not br.empty:
        fig, ax = plt.subplots(figsize=(6, 2.8))
        for k, m in enumerate(ms):
            d = br[br.method == m]
            ax.scatter(np.full(len(d), k) + (d.agent.to_numpy() - 3) * 0.08, d.gain, color=COLOR[m], s=14)
            ax.hlines(d.gain.mean(), k - 0.3, k + 0.3, color="#2b2b29", lw=1.5)
        ax.axhline(0, color="#8a8984", lw=0.8)
        ax.set_xticks(range(len(ms)), [NAMES[m] for m in ms], rotation=20, fontsize=7)
        ax.set_ylabel("J_k(BR) − J_k(π)")
        ax.set_title("Per-agent best-response gain (agents 0, 1, 5; all seeds)", fontsize=9)
        save(fig, "best_response_gains")
    attention_figure()


def attention_figure():
    import networkx as nx

    from .envs.roadnet import load_roadnet
    cands = sorted((RES / "traffic").glob("nashgat/seed*/attention.npz"))
    if not cands:
        return
    path = cands[0]
    z = np.load(path)
    net = load_roadnet(DATA_ROOT / "data/raw/colight/data/Jinan/3_4/roadnet_3_4.json")
    raw = json.loads((DATA_ROOT / "data/raw/colight/data/Jinan/3_4/roadnet_3_4.json").read_text())
    pos_all = {it["id"]: (it["point"]["x"], it["point"]["y"]) for it in raw["intersections"]}
    agents = net.agents
    steps = sorted({int(k.split("_")[0][1:]) for k in z.files})
    fig, axs = plt.subplots(1, len(steps), figsize=(3.6 * len(steps), 3.2))
    axs = np.atleast_1d(axs)
    for ax, t in zip(axs, steps):
        ei, al = z[f"t{t}_edges"], z[f"t{t}_alpha"]
        G = nx.DiGraph()
        pos = {k: pos_all[a] for k, a in enumerate(agents)}
        G.add_nodes_from(pos)
        self_w = np.zeros(len(agents))
        for (s, d), a in zip(ei.T, al):
            if s == d:
                self_w[d] = a
            else:
                G.add_edge(int(s), int(d), w=float(a))
        widths = [0.5 + 5 * G[u][v]["w"] for u, v in G.edges]
        nx.draw_networkx_nodes(G, pos, ax=ax, node_size=120 + 400 * self_w, node_color=PALETTE[4], edgecolors="#2b2b29",
                               linewidths=0.6)
        nx.draw_networkx_edges(G, pos, ax=ax, width=widths, edge_color=PALETTE[0], arrows=True, arrowsize=8,
                               connectionstyle="arc3,rad=0.15", node_size=200)
        ax.set_title(f"t = {t * 10} s: {G.number_of_edges()} active msgs", fontsize=8)
        ax.set_axis_off()
    fig.suptitle(f"NashGAT-MARL learned attention ({path.parent.name}, head mean). Edge width ∝ α_ij; node size ∝ self-attention",
                 fontsize=8)
    save(fig, "attention_graph")


def graphical_figures():
    s = RES / "graphical_games/final_profiles.csv"
    if not s.exists():
        return
    df = pd.read_csv(s)
    summ = df.groupby(["game", "objective", "lambda"]).agg(welfare=("welfare", "mean"), welfare_sd=("welfare", "std"),
                                                          nash_conv=("nash_conv", "mean"),
                                                          nash_conv_sd=("nash_conv", "std")).reset_index()
    summ.to_csv(TAB / "graphical_games_summary.csv", index=False)
    games = list(summ.game.unique())
    fig, axs = plt.subplots(1, len(games), figsize=(4.5 * len(games), 3.2))
    axs = np.atleast_1d(axs)
    sty = {"welfare_gap": (PALETTE[0], "o", "welfare + λ·gap (semi-grad.)"),
           "welfare_mix": (PALETTE[1], "x", "welfare + λ·own payoff (mixing)"),
           "welfare_fullgap": (PALETTE[2], "s", "welfare + λ·NashConv (full grad.)")}
    for ax, g in zip(axs, games):
        d = summ[summ.game == g]
        for obj, (c, mk, lab) in sty.items():
            e = d[d.objective == obj].sort_values("lambda")
            ax.plot(e.welfare, e.nash_conv, color=c, marker=mk, lw=1.5, ms=5, label=lab)
            for _, r in e.iterrows():
                if obj == "welfare_fullgap" and r["lambda"] in (0.5, 1.0, 2.0, 4.0):
                    ax.annotate(f"λ={r['lambda']:g}", (r.welfare, r.nash_conv), fontsize=6, xytext=(3, 3),
                                textcoords="offset points", color="#52514e")
        for obj, mk, lab in [("independent", "^", "independent grad."), ("welfare", "v", "welfare only")]:
            e = d[d.objective == obj]
            ax.scatter(e.welfare, e.nash_conv, marker=mk, color="#2b2b29", s=30, label=lab, zorder=4)
        ax.set(xlabel="social welfare Σ_i u_i", ylabel="NashConv (exact)", title=g)
    axs[0].legend(fontsize=6)
    save(fig, "graphical_games_frontier")

In [ ]:
import importlib, nashgat
importlib.invalidate_caches()
from nashgat.utils import set_seed
from nashgat.envs.roadnet import load_roadnet, load_flows, validate
from nashgat.envs.traffic_env import TrafficMAEnv, default_env_config
from dataclasses import replace
set_seed(0)
print("nashgat", nashgat.__version__, "imported from", Path(nashgat.__file__).parent)

### Dataset validation (real files)

In [ ]:
val = {}
for name, (rn, flows) in {
    "jinan_3x4": ("data/raw/colight/data/Jinan/3_4/roadnet_3_4.json", ["anon_3_4_jinan_real.json", "anon_3_4_jinan_real_2000.json", "anon_3_4_jinan_real_2500.json"]),
    "hangzhou_4x4": ("data/raw/colight/data/Hangzhou/4_4/roadnet_4_4.json", ["anon_4_4_hangzhou_real.json"]),
    "newyork_28x7": ("data/raw/colight/data/NewYork/28_7/roadnet_28_7.json", ["anon_28_7_newyork_real_double.json"]),
}.items():
    net = load_roadnet(DATA_ROOT / rn)
    for f in flows:
        v = validate(net, load_flows((DATA_ROOT / rn).parent / f))
        val[f] = {k: v[k] for k in ("n_signalised", "n_virtual", "n_roads", "n_phases", "n_movements", "n_vehicles",
                                     "departure_range_s", "invalid_routes")}
        assert v["invalid_routes"] == 0 and not v["issues"], (f, v["issues"])
display(pd.DataFrame(val).T)
assert val["anon_3_4_jinan_real.json"]["n_signalised"] == 12 and val["anon_3_4_jinan_real.json"]["n_vehicles"] == 6295
(RESULTS / "dataset_validation.json").write_text(json.dumps({"commit": commit, "files": val}, indent=1, default=str))

## Part 4 — AI patterns, demonstrated on running code
| Pattern | Kind | Role here |
|---|---|---|
| Perception–Decision–Action | architectural | `TrafficMAEnv._obs` → `Actor` → `env.step` |
| Actor–Critic | learning architecture | PPO actor + value/Q critic |
| CTDE | architectural | the critic sees global state at training; the actor sees only $h_i$ |
| Graph message passing | architectural/protocol | one GATv2 round over $E_t$ |
| Strategic best response | algorithmic | PPO best-response oracle for exploitability |
| Nash-aware optimisation | objective | gap regulariser (≡ reward mixing, Prop. 4) |
| Adaptive communication | protocol | Bernoulli gates and message penalty |

Patterns **not** used: experience replay (PPO is on-policy), BDI, Contract Net and MQTT. None has a functional role here.

In [ ]:
# Perception-Decision-Action + CTDE: one decision step; the actor input contains only local + neighbour messages
from nashgat.algos.mappo import MAPPOTrainer, make_config
env = TrafficMAEnv(replace(default_env_config(DATA_ROOT, "jinan", horizon=300), backend=BACKEND))
tr = MAPPOTrainer(env, make_config("nashgat"), seed=0)
obs, _ = env.reset()                                            # perception
o = torch.as_tensor(obs).unsqueeze(0)
mask = torch.as_tensor(tr.graph.active_mask()).unsqueeze(0)
ea = torch.as_tensor(env.edge_features()).unsqueeze(0)
logits, gate_logit, att = tr._actor_forward(tr.actor, o, mask, ea, return_attention=True)   # decision
a = logits[0].argmax(-1).numpy()
obs2, r, *_ = env.step(a)                                       # action
print("obs", obs.shape, "| logits", tuple(logits.shape), "| gates", tuple(gate_logit.shape), "| reward", r.round(2))
# CTDE check: changing a NON-neighbour's observation must not change agent 0's action distribution
nbrs0 = set(env.edge_index[0][env.edge_index[1] == 0].tolist())
far = [j for j in range(env.n_agents) if j not in nbrs0 | {0}][0]
o2 = o.clone(); o2[0, far] += 5.0
l2, _ = tr._actor_forward(tr.actor, o2, mask, ea)
assert torch.allclose(logits[0, 0], l2[0, 0]), "agent 0 must only depend on its 1-hop neighbourhood"
print(f"decentralisation check passed: agent 0 is unaffected by non-neighbour {far}")

In [ ]:
# Graph message passing: attention weights of agent 5 (an interior intersection) over its neighbours
ei, alpha = att[0].numpy(), att[1].mean(-1).detach().numpy()
sel = ei[1] == 5
print(pd.DataFrame({"sender": ei[0][sel], "receiver": ei[1][sel], "alpha(head mean)": alpha[sel].round(3)}))
assert abs(alpha[sel].sum() - 1.0) < 1e-5   # attention is a distribution over the neighbourhood (incl. self-loop)

In [ ]:
# Strategic best response + Nash equilibrium on 2x2 games (closed form)
from nashgat.games.matrix_games import prisoners_dilemma, matching_pennies, battle_of_sexes, pure_nash_equilibria, mixed_nash_2x2
C, D = np.array([1., 0.]), np.array([0., 1.])
pd_game = prisoners_dilemma()
print("PD pure NE:", pure_nash_equilibria(pd_game), "| NashConv(C,C) =", pd_game.nash_conv(C, C), "| NashConv(D,D) =", pd_game.nash_conv(D, D))
x, y = mixed_nash_2x2(matching_pennies()); print("Matching pennies mixed NE:", x, y, "NashConv:", matching_pennies().nash_conv(x, y))
x, y = mixed_nash_2x2(battle_of_sexes()); print("Battle of sexes mixed NE:", x.round(3), y.round(3))
assert pd_game.nash_conv(C, C) == 4.0 and abs(matching_pennies().nash_conv(x * 0 + .5, y * 0 + .5)) < 1e-12

In [ ]:
# Nash-aware optimisation: Prop. 4 checked numerically (gap semi-gradient == reward mixing)
from nashgat.games.polymatrix import networked_prisoners_dilemma, grid_anti_coordination, train_softmax_profile
g = networked_prisoners_dilemma(2, 2)
a_ = train_softmax_profile(g, "welfare_gap", lam=0.7, steps=200, seed=3)
b_ = train_softmax_profile(g, "welfare_mix", lam=0.7, steps=200, seed=3)
print("max |x_gap - x_mix| =", np.abs(a_["x"] - b_["x"]).max())
assert np.allclose(a_["x"], b_["x"], atol=1e-10)

In [ ]:
# Adaptive communication: gates switch off a sender's out-edges at the next step; failures drop links at random
from nashgat.graph import DynamicGraphBuilder
gb = DynamicGraphBuilder(env.edge_index, env.n_agents, gated=True, fail_prob=0.0)
gb.update(np.array([1, 0] * 6))
m = gb.active_mask()
print("active messages:", int(m.sum()), "of", m.size, "| per-agent sends:", gb.comm_cost(m).astype(int))
assert gb.comm_cost(m)[1::2].sum() == 0

### Part 5b — Unit tests (asserts) on the written package

In [ ]:
from nashgat.graph import batch_graphs
from nashgat.stats import holm, mean_ci
# simulator: conservation of vehicles and storage limits under random control
env_t = TrafficMAEnv(replace(default_env_config(DATA_ROOT, "jinan", horizon=600), backend="queuesim"))
env_t.reset(); rng = np.random.default_rng(0)
while True:
    _, r, _, tr_, info = env_t.step(rng.integers(0, env_t.n_actions, env_t.n_agents))
    s = env_t.sim
    assert s.next_dep == int((s.finish[:s.next_dep] >= 0).sum()) + int(s.count.sum()) + s.metrics()["entry_backlog"]
    assert (s.count <= s.cap).all() and (r <= 0).all()
    if tr_: break
# graph batching offsets
out, ea_ = batch_graphs(torch.tensor([[0, 1], [1, 0]]), 2, torch.tensor([[True, False], [True, True]]), torch.ones(2, 2, 3))
assert out.tolist() == [[0, 2, 3], [1, 3, 2]]
# NashConv of networked PD profiles
eye = np.eye(2); gpd = networked_prisoners_dilemma(2, 3)
assert gpd.pure_nash_equilibria() == [(1,) * 6] and gpd.nash_conv(eye[[1] * 6]) == 0
# every method performs one update
for mth in ["ippo", "mappo", "gnn_mappo", "nash_mappo", "nashgat", "nashgat_no_attention"]:
    t_ = MAPPOTrainer(TrafficMAEnv(replace(default_env_config(DATA_ROOT, "jinan", horizon=200), backend="queuesim")), make_config(mth), seed=0)
    h_ = t_.train(1)[-1]; assert np.isfinite(h_["loss_v"]) and h_["loss_gap"] >= 0
assert holm([0.01, 0.04, 0.03]) == [0.03, 0.06, 0.06] or np.allclose(holm([0.01, 0.04, 0.03]), [0.03, 0.06, 0.06])
print("all unit tests passed")

## Part 6 — Experiments
Every method uses **the same budget**: the same episodes, horizon, evaluation episodes and best-response budget. The hyper-parameters live in `AlgoConfig` and are saved with each run (`config.json`). Shared settings: hidden 64, 4 heads, actor learning rate 5e-4, critic learning rate 1e-3, γ = 0.99, GAE λ = 0.95, clip 0.2, 4 epochs × 4 minibatches, entropy 0.01. The `nashgat` default is λ = 0.5, μ = 0.05.

In [ ]:
from nashgat.algos.mappo import PRESETS, AlgoConfig
from dataclasses import asdict
display(pd.DataFrame({m: PRESETS[m] for m in METHODS + CFG["ablations"]}).T)
(RESULTS / "configs").mkdir(exist_ok=True)
(RESULTS / "configs/experiment_config.json").write_text(json.dumps({"mode": MODE, "cfg": CFG, "backend": BACKEND,
    "defaults": asdict(AlgoConfig()), "presets": PRESETS}, indent=1, default=str))

In [ ]:
from nashgat.experiment import run_single
from nashgat.rule_based import run_rule
torch.set_num_threads(max(1, os.cpu_count() // 2))
TRAFFIC = RESULTS / "traffic"
DRIVE = None
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive"); DRIVE = Path("/content/drive/MyDrive/nashgat_results"); DRIVE.mkdir(parents=True, exist_ok=True)

def checkpoint_to_drive():
    if DRIVE is not None:
        shutil.copytree(RESULTS, DRIVE / RESULTS.name, dirs_exist_ok=True)

t_start = time.time()
for seed in ([] if MODE == "REPRODUCE_PAPER" else CFG["seeds"]):   # REPRODUCE_PAPER: nothing is retrained
    for m in METHODS + CFG["ablations"]:
        if (TRAFFIC / m / f"seed{seed}" / "eval.json").exists():
            print("skip (exists)", m, seed); continue
        t0 = time.time()
        r = run_single(m, seed, DATA_ROOT, TRAFFIC, episodes=CFG["episodes"], horizon=CFG["horizon"], backend=BACKEND,
                       eval_episodes=CFG["eval_episodes"], robustness=CFG["robustness"], br_agents=CFG["br_agents"],
                       br_episodes=CFG["br_episodes"], br_eval_episodes=CFG["br_eval_episodes"], device=DEVICE,
                       log_every=max(1, CFG["episodes"] // 4))
        print(f"{m:24s} seed {seed}: ATT {r['eval']['nominal_stochastic']['avg_travel_time']:.1f} s  ({time.time()-t0:.0f}s)")
        checkpoint_to_drive()
print(f"total {time.time()-t_start:.0f}s")

In [ ]:
# non-learning references on the same simulator and scenarios
rows = []
scen = {"nominal": default_env_config(DATA_ROOT, "jinan", horizon=CFG["horizon"]),
        "transfer_hangzhou": default_env_config(DATA_ROOT, "hangzhou", horizon=CFG["horizon"])}
for sname, c in scen.items():
    for pol in ("fixed", "maxpressure"):
        rows.append({"scenario": sname, "controller": pol, **run_rule(TrafficMAEnv(replace(c, backend=BACKEND)), pol)})
rule_df = pd.DataFrame(rows)
if MODE == "REPRODUCE_PAPER":
    # keep the released file (it has more scenarios); the recomputed values must match it exactly (deterministic)
    rel = pd.read_csv(RESULTS / "rule_based.csv").set_index(["scenario", "controller"])
    for _, r in rule_df.iterrows():
        paper = rel.loc[(r.scenario, r.controller), "avg_travel_time"]
        print(f"{r.scenario:18s} {r.controller:12s} ATT recomputed {r.avg_travel_time:8.2f} | paper {paper:8.2f}")
        assert abs(r.avg_travel_time - paper) < 1e-6 * max(1, paper), "rule-based result does not match the paper"
    print("rule-based references: exact match with the paper")
else:
    rule_df.to_csv(RESULTS / "rule_based.csv", index=False)
display(rule_df[["scenario", "controller", "avg_travel_time", "avg_waiting_time", "throughput", "avg_queue_per_intersection"]])

In [ ]:
# REPRODUCE_PAPER: independent re-run of every released trained model (nominal Jinan scenario)
#  * greedy evaluation is deterministic  -> must equal the paper's value exactly
#  * stochastic evaluation (the paper's main metric) uses a new random stream -> must agree within sampling noise
from nashgat.algos.mappo import AlgoConfig
if MODE == "REPRODUCE_PAPER":
    torch.set_num_threads(1)
    chk = []
    for d in sorted(TRAFFIC.glob("*/seed*")):
        c_, ev_ = json.loads((d / "config.json").read_text()), json.loads((d / "eval.json").read_text())
        seed_ = int(d.name[4:])
        env_c = TrafficMAEnv(replace(default_env_config(DATA_ROOT, "jinan", "real", horizon=CFG["horizon"], seed=seed_), backend=BACKEND))
        tr_c = MAPPOTrainer(env_c, AlgoConfig(**c_["algo"]), seed=seed_); tr_c.load(d / "model.pt")
        set_seed(seed_); g_ = tr_c.evaluate(1, greedy=True)
        set_seed(seed_); s_ = tr_c.evaluate(CFG["eval_episodes"])
        chk.append({"run": d.parent.name, "seed": seed_,
                    "greedy_ATT_paper": ev_["nominal_greedy"]["avg_travel_time"], "greedy_ATT_rerun": g_["avg_travel_time"],
                    "stoch_ATT_paper": ev_["nominal_stochastic"]["avg_travel_time"], "stoch_ATT_rerun": s_["avg_travel_time"]})
    chk = pd.DataFrame(chk)
    chk["greedy_exact"] = np.isclose(chk.greedy_ATT_paper, chk.greedy_ATT_rerun, rtol=1e-6)
    chk["stoch_diff_%"] = 100 * (chk.stoch_ATT_rerun - chk.stoch_ATT_paper) / chk.stoch_ATT_paper
    chk.to_csv(RESULTS / "reproduction_check.csv", index=False)
    display(chk.round(2))
    print(f"greedy evaluations matching the paper exactly: {chk.greedy_exact.sum()} / {len(chk)}")
    print(f"stochastic ATT, re-run vs paper: median |diff| {chk['stoch_diff_%'].abs().median():.2f}%, max {chk['stoch_diff_%'].abs().max():.2f}%")
    if not chk.greedy_exact.all():
        print("NOTE: a non-exact greedy value means a different floating-point platform changed an action somewhere; "
              "check the size of the difference.")

In [ ]:
# exact graphical-game study (closed-form exploitability; independent of the simulator)
if MODE == "REPRODUCE_PAPER":
    # released profiles (scripts/run_graphical_games.py); Prop. 4 was re-checked numerically in Part 4
    gg = pd.read_csv(RESULTS / "graphical_games/final_profiles.csv")
else:
    gg_rows = []
    for gname, gm in {"npd_3x4": networked_prisoners_dilemma(3, 4), "anticoord_3x4": grid_anti_coordination(3, 4)}.items():
        for obj, lam in [("independent", 0.), ("welfare", 0.)] + [(o, l) for o in ("welfare_gap", "welfare_mix", "welfare_fullgap") for l in (0.25, 1.0, 4.0)]:
            for s in range(CFG["gg_seeds"]):
                res = train_softmax_profile(gm, obj, lam=lam, steps=CFG["gg_steps"], seed=s, init_scale=1.0)
                gg_rows.append({"game": gname, "objective": obj, "lambda": lam, "seed": s, "welfare": res["welfare"],
                                "nash_conv": res["nash_conv"], "p_action0_mean": float(res["x"][:, 0].mean())})
    (RESULTS / "graphical_games").mkdir(exist_ok=True)
    gg = pd.DataFrame(gg_rows); gg.to_csv(RESULTS / "graphical_games/final_profiles.csv", index=False)
display(gg.groupby(["game", "objective", "lambda"])[["welfare", "nash_conv"]].mean().round(3))

## Part 7 — Visualisations
Every figure is generated from the CSV/JSON files written above. The figures are training curves (team return, ATT, exploitability gap), congestion bars with per-seed points, welfare against measured exploitability, communication against congestion, robustness, ablations, best-response gains, learned attention on the real Jinan geometry, and the graphical-game frontier.

In [ ]:
from nashgat import report
FIGS = RESULTS / "figures"
report.configure(RESULTS, FIGS, DATA_ROOT)
report.main()
from IPython.display import Image, display as _d
for f in ["training_curves", "congestion_comparison", "tradeoffs", "best_response_gains", "robustness", "ablations",
          "attention_graph", "graphical_games_frontier"]:
    p = FIGS / f"{f}.png"
    if p.exists():
        print(f); _d(Image(filename=str(p), width=900))

In [ ]:
# REPRODUCE_PAPER: the tables regenerated in this session must equal the released (paper) tables
if MODE == "REPRODUCE_PAPER":
    for t in ["summary_all_methods.csv", "stat_tests_nashgat_vs_baselines.csv", "robustness.csv"]:
        new, old = pd.read_csv(RESULTS / "tables" / t), pd.read_csv(RELEASED / "results/tables" / t)
        num = [c for c in old.select_dtypes("number").columns if c in new]
        assert new.shape == old.shape and np.allclose(new[num].to_numpy(float), old[num].to_numpy(float), equal_nan=True, rtol=1e-9), t
        print(f"{t:40s} identical to the released table ({len(old)} rows)")
    S_ = pd.read_csv(RESULTS / "tables/summary_all_methods.csv")
    print("\nPaper main table (QueueSim, Jinan 3x4, 5 seeds, stochastic evaluation):")
    display(S_[S_.method.isin(METHODS)][["name", "n", "avg_travel_time_mean", "avg_travel_time_ci_lo", "avg_travel_time_ci_hi",
                                         "nash_conv_sampled_mean", "comm_per_agent_step_mean"]].round(2))

In [ ]:
# agent communication graph actually used by NashGAT over one evaluation episode (message frequency per edge)
import matplotlib.pyplot as plt, networkx as nx
from nashgat.algos.mappo import MAPPOTrainer
ck = sorted(TRAFFIC.glob("nashgat/seed*/model.pt"))[0]
env_v = TrafficMAEnv(replace(default_env_config(DATA_ROOT, "jinan", horizon=CFG["horizon"]), backend=BACKEND))
trv = MAPPOTrainer(env_v, make_config("nashgat"), seed=0); trv.load(ck)
b = trv.rollout()
freq = b["mask"].mean(0)
raw = json.loads((DATA_ROOT / "data/raw/colight/data/Jinan/3_4/roadnet_3_4.json").read_text())
xy = {it["id"]: (it["point"]["x"], it["point"]["y"]) for it in raw["intersections"]}
pos = {k: xy[a] for k, a in enumerate(env_v.agents)}
G = nx.DiGraph(); G.add_nodes_from(pos)
for (s_, d_), f_ in zip(env_v.edge_index.T, freq): G.add_edge(int(s_), int(d_), f=float(f_))
fig, ax = plt.subplots(figsize=(5, 4))
nx.draw_networkx(G, pos, ax=ax, node_color="#e87ba4", node_size=300, font_size=7, width=[0.3 + 4 * G[u][v]["f"] for u, v in G.edges],
                 edge_color="#2a78d6", connectionstyle="arc3,rad=0.15")
ax.set_title(f"Message frequency per directed edge (mean {freq.mean():.2f})", fontsize=9); ax.set_axis_off()
fig.savefig(FIGS / "communication_graph.png", dpi=200, bbox_inches="tight"); plt.show()

## Part 8 — Scientific analysis (computed automatically)
The interpretation below is **generated from the numbers in this run**. In `FAST_DEMO` mode (1 seed, a handful of episodes) no statistical claim is warranted, and the code says so.

In [ ]:
S = pd.read_csv(RESULTS / "tables/summary_all_methods.csv")
cols = ["name", "n", "avg_travel_time_mean", "avg_travel_time_ci_lo", "avg_travel_time_ci_hi", "avg_waiting_time_mean",
        "throughput_mean", "J_sum_mean", "nash_conv_sampled_mean", "comm_per_agent_step_mean", "train_time_s_mean"]
display(S[[c for c in cols if c in S]].round(2))
st_path = RESULTS / "tables/stat_tests_nashgat_vs_baselines.csv"
ST = pd.read_csv(st_path) if st_path.exists() else pd.DataFrame()
if len(ST): display(ST.round(4))

In [ ]:
def interpret(S, ST, n_seeds):
    out = []
    main = S[S.method.isin(METHODS)].set_index("method")
    best = main.avg_travel_time_mean.idxmin()
    out.append(f"Lowest mean ATT: {main.loc[best, 'name']} ({main.loc[best, 'avg_travel_time_mean']:.1f} s, n={int(main.loc[best, 'n'])}).")
    if "nashgat" in main.index:
        for m in main.index:
            if m == "nashgat": continue
            rel = (main.loc["nashgat", "avg_travel_time_mean"] - main.loc[m, "avg_travel_time_mean"]) / main.loc[m, "avg_travel_time_mean"] * 100
            out.append(f"NashGAT vs {main.loc[m, 'name']}: ATT {rel:+.1f}% (negative = NashGAT better).")
    if "nash_conv_sampled_mean" in main:
        nc = main.nash_conv_sampled_mean.dropna()
        if len(nc): out.append("Sampled BR NashConv (lower = less exploitable): " + ", ".join(f"{main.loc[k, 'name']} {v:.2f}" for k, v in nc.items()) + ".")
    if "comm_per_agent_step_mean" in main:
        out.append("Messages/agent/step: " + ", ".join(f"{main.loc[k, 'name']} {v:.2f}" for k, v in main.comm_per_agent_step_mean.items()) + ".")
    if n_seeds < 3:
        out.append(f"CAUTION: only {n_seeds} seed(s) per method; no significance test is meaningful. Differences above are anecdotal.")
    elif len(ST):
        sig = ST[ST.p_welch_holm < 0.05]
        out.append(f"{len(sig)} of {len(ST)} NashGAT-vs-baseline comparisons are significant at 0.05 after Holm correction (Welch t-test).")
        for _, r in sig.iterrows():
            out.append(f"  - {r.metric} vs {r.vs}: {r['rel_change_%']:+.1f}% ({'better' if r.ref_better else 'worse'}), p_Holm={r.p_welch_holm:.3g}")
    return "\n".join(out)

analysis = interpret(S, ST, len(CFG["seeds"]))
print(analysis)
(RESULTS / "analysis.txt").write_text(analysis)

## Part 9 — Research paper export
This part writes the verified BibTeX, the configuration files and an IEEE-style LaTeX draft whose result tables are filled from this run's CSV files. Every interpretive sentence in the draft is marked for author revision.

In [ ]:
%%writefile references.bib
% references.bib -- NashGAT-MARL literature review
% Every entry below was verified on 2026-10-19 by fetching metadata from at least one of:
%   Crossref REST API (api.crossref.org/works/<DOI>), OpenAlex (api.openalex.org/works/doi:<DOI>),
%   DataCite (api.datacite.org/dois/10.48550/arXiv.<id>), arXiv export API, or the official
%   proceedings page (proceedings.neurips.cc, proceedings.mlr.press, jmlr.org, iclr.cc).
% No entry was added from memory alone. Fields not returned by those sources were omitted.
% Entries whose peer-review status is "preprint" are marked as such in the note field.

% =====================================================================
% A. Seed papers supplied by the project (all 6 DOIs resolved)
% =====================================================================

% Verified: Crossref (10.1109/TPAMI.2025.3634378) + OpenAlex W4416366467 (abstract)
@article{shi2026ddfg,
  author  = {Shi, Yuchen and Duan, Shihong and Xu, Cheng and Wang, Ran and Ye, Fangwen and Yuen, Chau},
  title   = {Dynamic Deep Factor Graph for Multi-Agent Reinforcement Learning},
  journal = {IEEE Transactions on Pattern Analysis and Machine Intelligence},
  year    = {2026},
  volume  = {48},
  number  = {3},
  pages   = {3417--3431},
  doi     = {10.1109/TPAMI.2025.3634378},
  url     = {https://doi.org/10.1109/TPAMI.2025.3634378}
}

% Verified: Crossref (10.1016/j.neunet.2025.108437) + OpenAlex W4417305198; abstract via Semantic Scholar
@article{peng2026gde,
  author  = {Peng, Kexing and Qi, Hanwen and Ma, Tinghuai},
  title   = {Graph based multi-agent reinforcement learning with evolutionary population for cooperation},
  journal = {Neural Networks},
  year    = {2026},
  volume  = {196},
  pages   = {108437},
  doi     = {10.1016/j.neunet.2025.108437},
  url     = {https://doi.org/10.1016/j.neunet.2025.108437}
}

% Verified: Crossref (10.1016/j.neunet.2026.108693) + OpenAlex W7128066690; abstract via Semantic Scholar
@article{gou2026gsmarl,
  author  = {Gou, Fandi and Du, Haikuo and Cai, Yunze},
  title   = {A graph-based safe reinforcement learning method for multi-agent cooperation},
  journal = {Neural Networks},
  year    = {2026},
  volume  = {199},
  pages   = {108693},
  doi     = {10.1016/j.neunet.2026.108693},
  url     = {https://doi.org/10.1016/j.neunet.2026.108693}
}

% Verified: Crossref (10.1016/j.automatica.2025.112342) + OpenAlex W4410418157; abstract NOT available from open metadata
@article{liu2025distributednash,
  author  = {Liu, Qiwei and Yan, Huaicheng and Chen, Kaitian and Wang, Meng and Li, Zhichen},
  title   = {Distributed {Nash} equilibrium solution for multi-agent game in adversarial environment: A reinforcement learning method},
  journal = {Automatica},
  year    = {2025},
  volume  = {178},
  pages   = {112342},
  doi     = {10.1016/j.automatica.2025.112342},
  url     = {https://doi.org/10.1016/j.automatica.2025.112342}
}

% Verified: Crossref (10.1109/TCSI.2025.3548900) + OpenAlex W4408399536; abstract via Semantic Scholar
@article{zhang2025graphicalgames,
  author  = {Zhang, Hanfang and Wang, Bing-Chang and Cao, Ying},
  title   = {Reinforcement Learning Solutions to Stochastic Multi-Agent Graphical Games With Multiplicative Noise},
  journal = {IEEE Transactions on Circuits and Systems I: Regular Papers},
  year    = {2025},
  volume  = {72},
  number  = {7},
  pages   = {3570--3583},
  doi     = {10.1109/TCSI.2025.3548900},
  url     = {https://doi.org/10.1109/TCSI.2025.3548900}
}

% Verified: Crossref (10.1007/s40747-026-02356-7, incl. abstract) + OpenAlex W7163892192
@article{kuang2026llmgncf,
  author  = {Kuang, Zihao and Xu, Zengmin and Wan, Linkai and Zhang, Chunjiong},
  title   = {{LLM}-guided graph neural coordination framework for cooperative multi-agent reinforcement learning},
  journal = {Complex \& Intelligent Systems},
  year    = {2026},
  volume  = {12},
  number  = {9},
  pages   = {225},
  doi     = {10.1007/s40747-026-02356-7},
  url     = {https://doi.org/10.1007/s40747-026-02356-7}
}

% =====================================================================
% B. Foundational MARL / GNN / game-theory references
% =====================================================================

% Verified: Crossref (10.52202/068431-1787, NeurIPS 35 proceedings) + OpenAlex W7133226216 + DataCite arXiv:2103.01955
@inproceedings{yu2022mappo,
  author    = {Yu, Chao and Velu, Akash and Vinitsky, Eugene and Gao, Jiaxuan and Wang, Yu and Bayen, Alexandre and Wu, Yi},
  title     = {The Surprising Effectiveness of {PPO} in Cooperative Multi-Agent Games},
  booktitle = {Advances in Neural Information Processing Systems 35 (NeurIPS 2022)},
  year      = {2022},
  pages     = {24611--24624},
  doi       = {10.52202/068431-1787},
  url       = {https://doi.org/10.52202/068431-1787},
  note      = {Datasets and Benchmarks track; arXiv:2103.01955}
}

% Verified: DataCite (10.48550/arXiv.1707.06347). PREPRINT (arXiv only)
@misc{schulman2017ppo,
  author        = {Schulman, John and Wolski, Filip and Dhariwal, Prafulla and Radford, Alec and Klimov, Oleg},
  title         = {Proximal Policy Optimization Algorithms},
  year          = {2017},
  eprint        = {1707.06347},
  archivePrefix = {arXiv},
  doi           = {10.48550/arXiv.1707.06347},
  url           = {https://doi.org/10.48550/arXiv.1707.06347},
  note          = {Preprint}
}

% Verified: DataCite (arXiv:1710.10903) + OpenAlex W2766453196 (authors, abstract) + web search (ICLR 2018 conference track; Cambridge repository record). OpenReview id from secondary source.
@inproceedings{velickovic2018gat,
  author    = {Veli{\v{c}}kovi{\'c}, Petar and Cucurull, Guillem and Casanova, Arantxa and Romero, Adriana and Li{\`o}, Pietro and Bengio, Yoshua},
  title     = {Graph Attention Networks},
  booktitle = {International Conference on Learning Representations (ICLR)},
  year      = {2018},
  url       = {https://openreview.net/forum?id=rJXMpikCZ},
  note      = {arXiv:1710.10903}
}

% Verified: DataCite (arXiv:1706.02275) + proceedings.neurips.cc page (NIPS 2017, hash 68a9750337a418a86fe06c1991a1d64c)
@inproceedings{lowe2017maddpg,
  author    = {Lowe, Ryan and Wu, Yi and Tamar, Aviv and Harb, Jean and Abbeel, Pieter and Mordatch, Igor},
  title     = {Multi-Agent Actor-Critic for Mixed Cooperative-Competitive Environments},
  booktitle = {Advances in Neural Information Processing Systems 30 (NIPS 2017)},
  year      = {2017},
  url       = {https://proceedings.neurips.cc/paper_files/paper/2017/hash/68a9750337a418a86fe06c1991a1d64c-Abstract.html},
  note      = {arXiv:1706.02275}
}

% Verified: DataCite (arXiv:1711.00832) + proceedings.neurips.cc page (NIPS 2017, hash 3323fe11e9595c09af38fe67567a9394)
@inproceedings{lanctot2017psro,
  author    = {Lanctot, Marc and Zambaldi, Vinicius and Gruslys, Audrunas and Lazaridou, Angeliki and Tuyls, Karl and P{\'e}rolat, Julien and Silver, David and Graepel, Thore},
  title     = {A Unified Game-Theoretic Approach to Multiagent Reinforcement Learning},
  booktitle = {Advances in Neural Information Processing Systems 30 (NIPS 2017)},
  year      = {2017},
  url       = {https://proceedings.neurips.cc/paper_files/paper/2017/hash/3323fe11e9595c09af38fe67567a9394-Abstract.html},
  note      = {arXiv:1711.00832}
}

% Verified: DataCite (arXiv:1810.09202) + OpenAlex W2975343435 + iclr.cc/virtual/2020/poster/1548 (ICLR 2020 main conference poster)
@inproceedings{jiang2020dgn,
  author    = {Jiang, Jiechuan and Dun, Chen and Huang, Tiejun and Lu, Zongqing},
  title     = {Graph Convolutional Reinforcement Learning},
  booktitle = {International Conference on Learning Representations (ICLR)},
  year      = {2020},
  url       = {https://openreview.net/forum?id=HkxdQkSYDB},
  note      = {arXiv:1810.09202}
}

% Verified: DataCite (arXiv:1812.09755) + OpenAlex W2906394285 + iclr.cc/virtual/2019/poster/770 (via web search)
@inproceedings{singh2019ic3net,
  author    = {Singh, Amanpreet and Jain, Tushar and Sukhbaatar, Sainbayar},
  title     = {Learning when to Communicate at Scale in Multiagent Cooperative and Competitive Tasks},
  booktitle = {International Conference on Learning Representations (ICLR)},
  year      = {2019},
  url       = {https://iclr.cc/virtual/2019/poster/770},
  note      = {arXiv:1812.09755}
}

% Verified: proceedings.mlr.press/v97/das19a.html (citation meta: ICML, pp. 1538-1546) + DataCite arXiv:1810.11187
@inproceedings{das2019tarmac,
  author    = {Das, Abhishek and Gervet, Th{\'e}ophile and Romoff, Joshua and Batra, Dhruv and Parikh, Devi and Rabbat, Mike and Pineau, Joelle},
  title     = {{TarMAC}: Targeted Multi-Agent Communication},
  booktitle = {Proceedings of the 36th International Conference on Machine Learning (ICML)},
  series    = {Proceedings of Machine Learning Research},
  volume    = {97},
  pages     = {1538--1546},
  year      = {2019},
  url       = {https://proceedings.mlr.press/v97/das19a.html}
}

% Verified: DataCite (arXiv:2106.01969) + iclr.cc/virtual/2022/poster/7008 (ICLR 2022 poster page, WebFetch)
@inproceedings{leonardos2022mpg,
  author    = {Leonardos, Stefanos and Overman, Will and Panageas, Ioannis and Piliouras, Georgios},
  title     = {Global Convergence of Multi-Agent Policy Gradient in {Markov} Potential Games},
  booktitle = {International Conference on Learning Representations (ICLR)},
  year      = {2022},
  url       = {https://iclr.cc/virtual/2022/poster/7008},
  note      = {arXiv:2106.01969}
}

% Verified: DataCite (10.48550/arXiv.1301.2281) + arXiv API comment "Appears in Proceedings of ... UAI2001" + OpenAlex W2160135758
@inproceedings{kearns2001graphical,
  author    = {Kearns, Michael and Littman, Michael L. and Singh, Satinder},
  title     = {Graphical Models for Game Theory},
  booktitle = {Proceedings of the Seventeenth Conference on Uncertainty in Artificial Intelligence (UAI 2001)},
  year      = {2001},
  doi       = {10.48550/arXiv.1301.2281},
  url       = {https://doi.org/10.48550/arXiv.1301.2281}
}

% Verified: proceedings.mlr.press/v80/yang18d.html (citation meta: ICML, pp. 5571-5580) + DataCite arXiv:1802.05438
@inproceedings{yang2018meanfield,
  author    = {Yang, Yaodong and Luo, Rui and Li, Minne and Zhou, Ming and Zhang, Weinan and Wang, Jun},
  title     = {Mean Field Multi-Agent Reinforcement Learning},
  booktitle = {Proceedings of the 35th International Conference on Machine Learning (ICML)},
  series    = {Proceedings of Machine Learning Research},
  volume    = {80},
  pages     = {5571--5580},
  year      = {2018},
  url       = {https://proceedings.mlr.press/v80/yang18d.html}
}

% Verified: jmlr.org/papers/v4/hu03a.html (citation meta: JMLR vol 4, pp. 1039-1069) + OpenAlex W2120846115 (no DOI exists)
@article{hu2003nashq,
  author  = {Hu, Junling and Wellman, Michael P.},
  title   = {Nash {Q}-Learning for General-Sum Stochastic Games},
  journal = {Journal of Machine Learning Research},
  year    = {2003},
  volume  = {4},
  pages   = {1039--1069},
  url     = {https://www.jmlr.org/papers/v4/hu03a.html}
}

% Verified: Crossref (10.1137/070699652)
@article{daskalakis2009complexity,
  author  = {Daskalakis, Constantinos and Goldberg, Paul W. and Papadimitriou, Christos H.},
  title   = {The Complexity of Computing a {Nash} Equilibrium},
  journal = {SIAM Journal on Computing},
  year    = {2009},
  volume  = {39},
  number  = {1},
  pages   = {195--259},
  doi     = {10.1137/070699652},
  url     = {https://doi.org/10.1137/070699652}
}

% Verified: Crossref (10.24963/ijcai.2019/66)
@inproceedings{lockhart2019exploitability,
  author    = {Lockhart, Edward and Lanctot, Marc and P{\'e}rolat, Julien and Lespiau, Jean-Baptiste and Morrill, Dustin and Timbers, Finbarr and Tuyls, Karl},
  title     = {Computing Approximate Equilibria in Sequential Adversarial Games by Exploitability Descent},
  booktitle = {Proceedings of the Twenty-Eighth International Joint Conference on Artificial Intelligence (IJCAI)},
  year      = {2019},
  pages     = {464--470},
  doi       = {10.24963/ijcai.2019/66},
  url       = {https://doi.org/10.24963/ijcai.2019/66}
}

% Verified: Crossref (10.1126/science.add4679)
@article{perolat2022stratego,
  author  = {Perolat, Julien and De Vylder, Bart and Hennes, Daniel and Tarassov, Eugene and Strub, Florian and de Boer, Vincent and Muller, Paul and Connor, Jerome T. and Burch, Neil and Anthony, Thomas and McAleer, Stephen and Elie, Romuald and Cen, Sarah H. and Wang, Zhe and Gruslys, Audrunas and Malysheva, Aleksandra and Khan, Mina and Ozair, Sherjil and Timbers, Finbarr and Pohlen, Toby and Eccles, Tom and Rowland, Mark and Lanctot, Marc and Lespiau, Jean-Baptiste and Piot, Bilal and Omidshafiei, Shayegan and Lockhart, Edward and Sifre, Laurent and Beauguerlange, Nathalie and Munos, Remi and Silver, David and Singh, Satinder and Hassabis, Demis and Tuyls, Karl},
  title   = {Mastering the game of {Stratego} with model-free multiagent reinforcement learning},
  journal = {Science},
  year    = {2022},
  volume  = {378},
  number  = {6623},
  pages   = {990--996},
  doi     = {10.1126/science.add4679},
  url     = {https://doi.org/10.1126/science.add4679}
}

% Verified: DataCite (10.48550/arXiv.1603.01121). PREPRINT (arXiv only)
@misc{heinrich2016nfsp,
  author        = {Heinrich, Johannes and Silver, David},
  title         = {Deep Reinforcement Learning from Self-Play in Imperfect-Information Games},
  year          = {2016},
  eprint        = {1603.01121},
  archivePrefix = {arXiv},
  doi           = {10.48550/arXiv.1603.01121},
  url           = {https://doi.org/10.48550/arXiv.1603.01121},
  note          = {Preprint}
}

% Verified: OpenAlex (10.48550/arXiv.2303.04865, W4323921909). PREPRINT (arXiv; no peer-reviewed version verified)
@misc{zhou2023networkedmpg,
  author        = {Zhou, Zhaoyi and Chen, Zaiwei and Lin, Yiheng and Wierman, Adam},
  title         = {Convergence Rates for Localized Actor-Critic in Networked {Markov} Potential Games},
  year          = {2023},
  eprint        = {2303.04865},
  archivePrefix = {arXiv},
  doi           = {10.48550/arXiv.2303.04865},
  url           = {https://doi.org/10.48550/arXiv.2303.04865},
  note          = {Preprint}
}

% Verified: Crossref (10.52202/075280-1624, NeurIPS 36 proceedings) + OpenAlex W7133200960 (abstract)
@inproceedings{park2023networkedzerosum,
  author    = {Park, Chanwoo and Zhang, Kaiqing and Ozdaglar, Asuman},
  title     = {Multi-Player Zero-Sum {Markov} Games with Networked Separable Interactions},
  booktitle = {Advances in Neural Information Processing Systems 36 (NeurIPS 2023)},
  year      = {2023},
  pages     = {37354--37369},
  doi       = {10.52202/075280-1624},
  url       = {https://doi.org/10.52202/075280-1624}
}

% Verified: Crossref (10.1109/tsmc.2024.3462762) + OpenAlex W4403094994 (abstract)
@article{moghaddam2025nashseeking,
  author  = {Ramezani Moghaddam, Alireza and Kebriaei, Hamed},
  title   = {Multiagent Reinforcement Learning for {Nash} Equilibrium Seeking in General-Sum {Markov} Games},
  journal = {IEEE Transactions on Systems, Man, and Cybernetics: Systems},
  year    = {2025},
  volume  = {55},
  number  = {1},
  pages   = {221--227},
  doi     = {10.1109/TSMC.2024.3462762},
  url     = {https://doi.org/10.1109/TSMC.2024.3462762}
}

% =====================================================================
% C. Traffic signal control: simulator, classic RL/GNN baselines
% =====================================================================

% Verified: Crossref (10.1145/3357384.3357902) + OpenAlex W2983178256 (abstract) + DataCite arXiv:1905.05717 (full title)
% Verification note: Crossref title field records only "CoLight"; full title verified via DataCite record of arXiv:1905.05717
@inproceedings{wei2019colight,
  author    = {Wei, Hua and Xu, Nan and Zhang, Huichu and Zheng, Guanjie and Zang, Xinshi and Chen, Chacha and Zhang, Weinan and Zhu, Yanmin and Xu, Kai and Li, Zhenhui},
  title     = {{CoLight}: Learning Network-level Cooperation for Traffic Signal Control},
  booktitle = {Proceedings of the 28th ACM International Conference on Information and Knowledge Management (CIKM)},
  year      = {2019},
  pages     = {1913--1922},
  doi       = {10.1145/3357384.3357902},
  url       = {https://doi.org/10.1145/3357384.3357902},
}

% Verified: Crossref (10.1145/3308558.3314139) + OpenAlex W2945991855 (abstract)
@inproceedings{zhang2019cityflow,
  author    = {Zhang, Huichu and Feng, Siyuan and Liu, Chang and Ding, Yaoyao and Zhu, Yichen and Zhou, Zihan and Zhang, Weinan and Yu, Yong and Jin, Haiming and Li, Zhenhui},
  title     = {{CityFlow}: A Multi-Agent Reinforcement Learning Environment for Large Scale City Traffic Scenario},
  booktitle = {The World Wide Web Conference (WWW '19)},
  year      = {2019},
  pages     = {3620--3624},
  doi       = {10.1145/3308558.3314139},
  url       = {https://doi.org/10.1145/3308558.3314139}
}

% Verified: Crossref (10.1145/3292500.3330949) + OpenAlex W2964749398 (abstract) + web search (pure.psu.edu, full title)
% Verification note: Crossref title field records only "PressLight"; full title confirmed via Penn State research portal (pure.psu.edu) record
@inproceedings{wei2019presslight,
  author    = {Wei, Hua and Chen, Chacha and Zheng, Guanjie and Wu, Kan and Gayah, Vikash and Xu, Kai and Li, Zhenhui},
  title     = {{PressLight}: Learning Max Pressure Control to Coordinate Traffic Signals in Arterial Network},
  booktitle = {Proceedings of the 25th ACM SIGKDD International Conference on Knowledge Discovery \& Data Mining (KDD)},
  year      = {2019},
  pages     = {1290--1298},
  doi       = {10.1145/3292500.3330949},
  url       = {https://doi.org/10.1145/3292500.3330949},
}

% Verified: Crossref (10.1609/aaai.v34i04.5744, incl. abstract)
@inproceedings{chen2020mplight,
  author    = {Chen, Chacha and Wei, Hua and Xu, Nan and Zheng, Guanjie and Yang, Ming and Xiong, Yuanhao and Xu, Kai and Li, Zhenhui},
  title     = {Toward A Thousand Lights: Decentralized Deep Reinforcement Learning for Large-Scale Traffic Signal Control},
  booktitle = {Proceedings of the AAAI Conference on Artificial Intelligence},
  year      = {2020},
  volume    = {34},
  number    = {04},
  pages     = {3414--3421},
  doi       = {10.1609/aaai.v34i04.5744},
  url       = {https://doi.org/10.1609/aaai.v34i04.5744}
}

% Verified: Crossref (10.1109/TITS.2019.2901791) + OpenAlex W2915117209 (abstract)
@article{chu2020marltsc,
  author  = {Chu, Tianshu and Wang, Jie and Codec{\`a}, Lara and Li, Zhaojian},
  title   = {Multi-Agent Deep Reinforcement Learning for Large-Scale Traffic Signal Control},
  journal = {IEEE Transactions on Intelligent Transportation Systems},
  year    = {2020},
  volume  = {21},
  number  = {3},
  pages   = {1086--1095},
  doi     = {10.1109/TITS.2019.2901791},
  url     = {https://doi.org/10.1109/TITS.2019.2901791}
}

% Verified: Crossref (10.1145/3447556.3447565, incl. abstract)
% Verification note: Title as recorded by Crossref; the author PDF carries a longer subtitle not confirmed against the publisher page
@article{wei2021tscsurvey,
  author  = {Wei, Hua and Zheng, Guanjie and Gayah, Vikash and Li, Zhenhui},
  title   = {Recent Advances in Reinforcement Learning for Traffic Signal Control},
  journal = {ACM SIGKDD Explorations Newsletter},
  year    = {2021},
  volume  = {22},
  number  = {2},
  pages   = {12--18},
  doi     = {10.1145/3447556.3447565},
  url     = {https://doi.org/10.1145/3447556.3447565},
}

% =====================================================================
% D. Recent (2023-2026) graph-MARL traffic signal control
% =====================================================================

% Verified: Crossref (10.1109/jiot.2025.3525640) + OpenAlex W4406028115 (abstract)
@article{shang2025magsac,
  author  = {Shang, Jing and Meng, Shunmei and Hou, Jun and Zhao, Xiaoran and Zhou, Xiaokang and Jiang, Rong and Qi, Lianyong and Li, Qianmu},
  title   = {Graph-Based Cooperation Multiagent Reinforcement Learning for Intelligent Traffic Signal Control},
  journal = {IEEE Internet of Things Journal},
  year    = {2025},
  volume  = {12},
  number  = {10},
  pages   = {14362--14374},
  doi     = {10.1109/JIOT.2025.3525640},
  url     = {https://doi.org/10.1109/JIOT.2025.3525640}
}

% Verified: Crossref (10.1109/tits.2025.3526223) + OpenAlex W4406457539 (abstract)
@article{zhai2025hgatmarl,
  author  = {Zhai, Ziyang and Hao, Ruru and Cui, Boyang and Wang, Siyi},
  title   = {{HGAT} and Multi-Agent {RL}-Based Method for Multi-Intersection Traffic Signal Control},
  journal = {IEEE Transactions on Intelligent Transportation Systems},
  year    = {2025},
  volume  = {26},
  number  = {5},
  pages   = {6848--6864},
  doi     = {10.1109/TITS.2025.3526223},
  url     = {https://doi.org/10.1109/TITS.2025.3526223}
}

% Verified: Crossref (10.1109/tmc.2025.3556243) + OpenAlex W4409014172 (abstract)
@article{wang2025hypergraphtsc,
  author  = {Wang, Kang and Shen, Zhishu and Lei, Zhen and Liu, Xianhui and Zhang, Tiehua},
  title   = {Toward Multi-Agent Reinforcement Learning Based Traffic Signal Control Through Spatio-Temporal Hypergraphs},
  journal = {IEEE Transactions on Mobile Computing},
  year    = {2025},
  volume  = {24},
  number  = {9},
  pages   = {8258--8271},
  doi     = {10.1109/TMC.2025.3556243},
  url     = {https://doi.org/10.1109/TMC.2025.3556243}
}

% Verified: Crossref (10.1177/03611981241297979) + OpenAlex W4407058066 (abstract)
@article{yang2025gatddpg,
  author  = {Yang, Guoqing and Wen, Xin and Chen, Fuqiang},
  title   = {Multi-Agent Deep Reinforcement Learning with Graph Attention Network for Traffic Signal Control in Multiple-Intersection Urban Areas},
  journal = {Transportation Research Record: Journal of the Transportation Research Board},
  year    = {2025},
  volume  = {2679},
  number  = {4},
  pages   = {880--898},
  doi     = {10.1177/03611981241297979},
  url     = {https://doi.org/10.1177/03611981241297979}
}

% Verified: Crossref (10.1016/j.engappai.2025.111602) + OpenAlex W4412102495; abstract NOT available from open metadata
@article{yang2025storeforward,
  author  = {Yang, Kangkang and Wang, Zhiwen and Meng, Xinyou and Li, Long and Shi, Yaoke and Yu, Yuling and Wang, Haoxu and Yao, Ziheng},
  title   = {Store-and-forward with graph attention: Enhanced multi-agent reinforcement learning for emergency-responsive traffic signal control},
  journal = {Engineering Applications of Artificial Intelligence},
  year    = {2025},
  volume  = {159},
  pages   = {111602},
  doi     = {10.1016/j.engappai.2025.111602},
  url     = {https://doi.org/10.1016/j.engappai.2025.111602}
}

% Verified: Crossref (10.1016/j.eswa.2026.132707) + OpenAlex W7160396645; abstract NOT available from open metadata
@article{yang2026heterodist,
  author  = {Yang, Shantian and Zeng, Zheng},
  title   = {Heterogeneous graph distributional reinforcement learning for out-of-distribution traffic signal control},
  journal = {Expert Systems with Applications},
  year    = {2026},
  volume  = {325},
  pages   = {132707},
  doi     = {10.1016/j.eswa.2026.132707},
  url     = {https://doi.org/10.1016/j.eswa.2026.132707}
}

% Verified: Crossref (10.1038/s41598-026-65802-z) + OpenAlex W7202366197 (abstract). Crossref returned no volume/article number.
@article{zhang2026sanis,
  author  = {Zhang, Yongnan and Peng, Sirui and Luo, Xiaobin and Gu, Xin and Jia, Xiaolu and Li, Yunxuan},
  title   = {Distributed multi-agent reinforcement learning with neighborhood-aware spatio-temporal coordination for traffic network signal control},
  journal = {Scientific Reports},
  year    = {2026},
  doi     = {10.1038/s41598-026-65802-z},
  url     = {https://doi.org/10.1038/s41598-026-65802-z}
}

% Verified: Crossref (10.1016/j.asoc.2026.116565; Crossref issue date January 2027) + OpenAlex W7215037705; abstract NOT available from open metadata
@article{ji2027stackelbergtsc,
  author  = {Ji, Ang and Gao, Yelin and Sun, Zhanbo and Cai, Yiming and Lin, Xia and Liu, Lin and Wang, Wenjun},
  title   = {Asynchronous multi-agent reinforcement learning for network-wide traffic signal control via {Stackelberg} game prioritization},
  journal = {Applied Soft Computing},
  year    = {2027},
  volume  = {204},
  pages   = {116565},
  doi     = {10.1016/j.asoc.2026.116565},
  url     = {https://doi.org/10.1016/j.asoc.2026.116565}
}

% Verified: Crossref (10.1109/itsc57777.2023.10422534) + OpenAlex W4391770468 (abstract)
@inproceedings{zhang2023nashdqn,
  author    = {Zhang, Yuli and Wang, Shangbo and Ma, Xiaoguang and Yue, Wenwei and Jiang, Ruiyuan},
  title     = {Large-Scale Traffic Signal Control by a {Nash} Deep {Q}-network Approach},
  booktitle = {2023 IEEE 26th International Conference on Intelligent Transportation Systems (ITSC)},
  year      = {2023},
  pages     = {4584--4591},
  doi       = {10.1109/ITSC57777.2023.10422534},
  url       = {https://doi.org/10.1109/ITSC57777.2023.10422534}
}

% Verified: Crossref (10.3390/electronics13101910) + OpenAlex W4396858723 (abstract)
@article{yan2024dcnpg,
  author  = {Yan, Liping and Wang, Jing},
  title   = {Deep Reinforcement Learning for Ecological and Distributed Urban Traffic Signal Control with Multi-Agent Equilibrium Decision Making},
  journal = {Electronics},
  year    = {2024},
  volume  = {13},
  number  = {10},
  pages   = {1910},
  doi     = {10.3390/electronics13101910},
  url     = {https://doi.org/10.3390/electronics13101910}
}

% =====================================================================
% E. Communication-aware / dynamic-graph MARL (2024-2026)
% =====================================================================

% Verified: Crossref (10.1609/aaai.v39i22.34507) + OpenAlex W4409348008 (abstract)
@inproceedings{zhang2025tgcnet,
  author    = {Zhang, Zhuohui and He, Bin and Cheng, Bin and Li, Gang},
  title     = {Bridging Training and Execution via Dynamic Directed Graph-Based Communication in Cooperative Multi-Agent Systems},
  booktitle = {Proceedings of the AAAI Conference on Artificial Intelligence},
  year      = {2025},
  volume    = {39},
  number    = {22},
  pages     = {23395--23403},
  doi       = {10.1609/aaai.v39i22.34507},
  url       = {https://doi.org/10.1609/aaai.v39i22.34507}
}

% Verified: Crossref (10.1109/tmlcn.2026.3719222) + OpenAlex W4416236299 (abstract)
@article{zhang2026commprotocols,
  author  = {Zhang, Xinren and Yu, Jiadong and Zhong, Zixin},
  title   = {Learning Efficient Communication Protocols for Multi-Agent Reinforcement Learning},
  journal = {IEEE Transactions on Machine Learning in Communications and Networking},
  year    = {2026},
  volume  = {4},
  pages   = {1335--1352},
  doi     = {10.1109/TMLCN.2026.3719222},
  url     = {https://doi.org/10.1109/TMLCN.2026.3719222}
}

% Verified: OpenAlex (10.48550/arXiv.2501.00165, W4406030907). PREPRINT (arXiv only)
@misc{mcclusky2024dyngraphcomm,
  author        = {McClusky, Ben},
  title         = {Dynamic Graph Communication for Decentralised Multi-Agent Reinforcement Learning},
  year          = {2024},
  eprint        = {2501.00165},
  archivePrefix = {arXiv},
  doi           = {10.48550/arXiv.2501.00165},
  url           = {https://doi.org/10.48550/arXiv.2501.00165},
  note          = {Preprint}
}

% =====================================================================
% F. Closest prior work for the novelty check (graph attention + Nash / game-theoretic MARL)
% =====================================================================

% Verified: Crossref (10.5267/j.ijdns.2025.9.009) + OpenAlex W7104177392 (abstract)
@article{alqudah2026nashhgt,
  author  = {Al-Qudah, Rabiah and Al Tawil, Arar and AlMuhajiri, Mrouj and Almazaydeh, Laiali and Suen, Ching Y.},
  title   = {{Nash}-regularized heterogeneous graph transformer networks for strategic task offloading in {6G} edge computing},
  journal = {International Journal of Data and Network Science},
  year    = {2026},
  volume  = {10},
  number  = {1},
  pages   = {415--432},
  doi     = {10.5267/j.ijdns.2025.9.009},
  url     = {https://doi.org/10.5267/j.ijdns.2025.9.009}
}

% Verified: Crossref (10.1088/2631-8695/ae67d6) + OpenAlex W7160310166 (abstract)
@article{fei2026gatgame,
  author  = {Fei, Qiang and Zhang, Wenjian and Liu, Jialin},
  title   = {Graph attention-based distributed game-theoretic framework for elastic collaborative predictive maintenance in communication-constrained production line clusters},
  journal = {Engineering Research Express},
  year    = {2026},
  volume  = {8},
  number  = {9},
  pages   = {095244},
  doi     = {10.1088/2631-8695/ae67d6},
  url     = {https://doi.org/10.1088/2631-8695/ae67d6}
}

% Verified: Crossref (10.1109/tsg.2026.3669058) + OpenAlex W7131873644 (abstract)
@article{li2026evpricing,
  author  = {Li, Yujing and Xing, Qiang and Lv, Si and Li, Zening},
  title   = {Mixed Competitive--Cooperative Pricing for {EV} Charging Stations: A Multi-Agent Reinforcement Learning Approach With Heterogeneous Hierarchical Attention},
  journal = {IEEE Transactions on Smart Grid},
  year    = {2026},
  volume  = {17},
  number  = {5},
  pages   = {4398--4411},
  doi     = {10.1109/TSG.2026.3669058},
  url     = {https://doi.org/10.1109/TSG.2026.3669058}
}

% Verified: Crossref (10.1016/j.inffus.2026.104506) + OpenAlex W7163079619; abstract NOT available from open metadata
@article{cheng2026sgasac,
  author  = {Cheng, Xinmin and Xu, Binxin and Yin, Hongwei and Shen, Shigen and Liu, Zhiquan and Wang, Tian and Li, Ruidong},
  title   = {{SGA-SAC}: Fusing {Bayesian} {Stackelberg} games to {DRL} with graph attention for explainable task offloading in vehicular edge computing},
  journal = {Information Fusion},
  year    = {2026},
  volume  = {136},
  pages   = {104506},
  doi     = {10.1016/j.inffus.2026.104506},
  url     = {https://doi.org/10.1016/j.inffus.2026.104506}
}

% Verified: Crossref (10.21203/rs.3.rs-10945491/v1, type posted-content) + OpenAlex W7211970186. PREPRINT (Research Square); abstract NOT available from open metadata
@misc{samanta2026gatmappoeig,
  author = {Samanta, Sukanya},
  title  = {{GAT-MAPPO-EIG}: A Graph Attention Multi-Agent Reinforcement Learning Framework for Escape Interdiction Games on Dynamic Transportation Networks},
  year   = {2026},
  howpublished = {Research Square preprint},
  doi    = {10.21203/rs.3.rs-10945491/v1},
  url    = {https://doi.org/10.21203/rs.3.rs-10945491/v1},
  note   = {Preprint, not peer reviewed}
}

% =====================================================================
% Added during mathematical modelling (main session, 2026-10-19)
% =====================================================================

% Verified: OpenAlex search (title, authors Kakade & Langford, year 2002; landing page CiteSeerX 10.1.1.121.3684).
% Venue (ICML 2002) not returned by OpenAlex -> requires verification before submission.
@inproceedings{kakade2002approximately,
  author    = {Kakade, Sham M. and Langford, John C.},
  title     = {Approximately Optimal Approximate Reinforcement Learning},
  booktitle = {Proceedings of the International Conference on Machine Learning (venue requires verification)},
  year      = {2002},
  url       = {http://citeseerx.ist.psu.edu/viewdoc/summary?doi=10.1.1.121.3684}
}

% Verified: Crossref (10.52202/068431-0140), container-title "Advances in Neural Information Processing Systems 35", 2022
@inproceedings{zhang2022softmaxmpg,
  author    = {Zhang, Runyu and Mei, Jincheng and Dai, Bo and Schuurmans, Dale and Li, Na},
  title     = {On the Global Convergence Rates of Decentralized Softmax Gradient Play in Markov Potential Games},
  booktitle = {Advances in Neural Information Processing Systems 35},
  year      = {2022},
  doi       = {10.52202/068431-0140}
}

% Verified: DataCite (10.48550/arXiv.2105.14491): title, creators Brody/Alon/Yahav, year 2021 (arXiv record).
% Peer-reviewed venue (ICLR 2022) not confirmed in this session -> cited as arXiv preprint.
@misc{brody2021gatv2,
  author = {Brody, Shaked and Alon, Uri and Yahav, Eran},
  title  = {How Attentive are Graph Attention Networks?},
  year   = {2021},
  doi    = {10.48550/arXiv.2105.14491},
  note   = {arXiv:2105.14491 (preprint record)}
}

% Verified: DataCite (10.48550/arXiv.1506.02438): title, creators, year 2015 (arXiv record).
@misc{schulman2015gae,
  author = {Schulman, John and Moritz, Philipp and Levine, Sergey and Jordan, Michael and Abbeel, Pieter},
  title  = {High-Dimensional Continuous Control Using Generalized Advantage Estimation},
  year   = {2015},
  doi    = {10.48550/arXiv.1506.02438},
  note   = {arXiv:1506.02438 (preprint record)}
}

% Verified: OpenAlex (doi 10.2307/4615733): title, author, venue, volume 6, pages 65-70, year 1979.
@article{holm1979,
  author  = {Holm, Sture},
  title   = {A Simple Sequentially Rejective Multiple Test Procedure},
  journal = {Scandinavian Journal of Statistics},
  year    = {1979},
  volume  = {6},
  pages   = {65--70},
  doi     = {10.2307/4615733}
}

In [ ]:
shutil.copy("references.bib", RESULTS / "references.bib")
main_tab = S[S.method.isin(METHODS)]
def fmt(r, c):
    m, lo, hi = r.get(c + "_mean"), r.get(c + "_ci_lo"), r.get(c + "_ci_hi")
    if pd.isna(m): return "--"
    return f"{m:.1f}" if pd.isna(lo) else f"{m:.1f} [{lo:.1f}, {hi:.1f}]"
rows_tex = "\n".join(f"{r['name']} & {fmt(r, 'avg_travel_time')} & {fmt(r, 'avg_waiting_time')} & {fmt(r, 'throughput')} & "
                     f"{fmt(r, 'nash_conv_sampled')} & {fmt(r, 'comm_per_agent_step')} \\\\" for _, r in main_tab.iterrows())
tex = rf"""\documentclass[journal]{{IEEEtran}}
\usepackage{{amsmath,amssymb,graphicx,booktabs}}
\begin{{document}}
\title{{Exploitability-Aware Graph-Attention MARL for Traffic-Signal Control: Draft generated from run ({MODE}, backend={BACKEND})}}
\author{{Badr El Khalyly and Zouhair Ibn Batouta}}
\maketitle
\begin{{abstract}}
AUTHOR: write from the analysis below. Auto-generated facts: {len(CFG['seeds'])} seed(s), {CFG['episodes']} episodes, horizon {CFG['horizon']} s, simulator {BACKEND}.
\end{{abstract}}
\section{{Results}}
\begin{{table}}[t]\centering\caption{{Nominal Jinan 3$\times$4 results (mean [95\% CI] over seeds; simulator: {BACKEND}).}}
\resizebox{{\columnwidth}}{{!}}{{\begin{{tabular}}{{lccccc}}\toprule
Method & ATT (s) & Wait (s) & Throughput & BR NashConv & Msgs/agent \\ \midrule
{rows_tex}
\bottomrule\end{{tabular}}}}\end{{table}}
\begin{{figure}}[t]\centering\includegraphics[width=\columnwidth]{{figures/training_curves.pdf}}\caption{{Training curves.}}\end{{figure}}
\noindent Auto-generated analysis (verify before use): \begin{{verbatim}}
{analysis}
\end{{verbatim}}
\bibliographystyle{{IEEEtran}}\nocite{{wei2019colight,yu2022mappo,velickovic2018gat,zhang2019cityflow}}\bibliography{{references}}
\end{{document}}
"""
(RESULTS / "manuscript_draft.tex").write_text(tex, encoding="utf-8")
print("wrote", RESULTS / "manuscript_draft.tex")

In [ ]:
# bundle every artefact (figures, CSV tables, configs, checkpoints, BibTeX, LaTeX draft) and download
archive = shutil.make_archive(str(RESULTS.parent / "nashgat_results"), "zip", RESULTS)
print("archive:", archive, f"({Path(archive).stat().st_size / 1e6:.1f} MB)")
checkpoint_to_drive()
if IN_COLAB:
    from google.colab import files
    files.download(archive)

### Cells needing Colab-only resources
* The CityFlow build cell needs Linux, cmake and a C++ compiler. Outside Colab it reports that CityFlow is unavailable and falls back to **QueueSim**.
* Google Drive mounting (`USE_DRIVE=True`) and `files.download` exist only in Colab. Elsewhere they are skipped.